# Build a Large Language Model From Scratch

#### 1 Understanding large language models

##### _1.1 What is an LLM?_

<div>
<img src="_docs/11-what-is-an-llm.png" width="800"/>
</div>

##### _1.2 Applications of LLMs_

<div>
<img src="_docs/12-applications-of-llms.png" width="800"/>
</div>

##### _1.3 Stages of building and using LLMs_

<div>
<img src="_docs/13-stages-of-building-and-using-llms.png" width="800"/>
</div>

##### _1.4 Introducing the transformer architecture_

<div>
<img src="_docs/14-introducing-transformer-architecture-01.png" width="800"/><br><br>
<img src="_docs/14-introducing-transformer-architecture-02.png" width="800"/><br><br>
<img src="_docs/14-introducing-transformer-architecture-03.png" width="800"/>
</div>

##### _1.5 Utilizing large datasets_

##### _1.6 A closer look at the GPT architecture_

<div>
<img src="_docs/16-closer-look-at-gpt-architecture.png" width="800"/>
</div>

##### _1.7 Building a large language model_

<div>
<img src="_docs/17-building-an-llm.png" width="800"/>
</div>


## STAGE 1 - Building an LLM

## Step #1 - Data preparation and sampling

#### 2 Working with text data

<div>
<img src="_docs/20-working-with-text-data.png" width="800"/>
</div>

##### _2.1 Understanding word embeddings_

<div>
<img src="_docs/21-understanding-word-embeddings-01.png" width="800"/><br><br>
<img src="_docs/21-understanding-word-embeddings-02.png" width="800"/>
</div>

##### _2.2 Tokenizing text_

<div>
<img src="_docs/22-tokenizing-text-01.png" width="800"/>
</div>

In [1]:
import os
import urllib

# Define the URL of the text file and the local file path to save it.
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
folder_path = "the_verdict"
file_path = os.path.join(folder_path, "the-verdict.txt")

# Create the folder if it doesn't exist.
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

# Download the text file from the URL and save it locally.
if not os.path.exists(file_path):
    print(f"Downloading '{file_path}' from '{url}'...")
    urllib.request.urlretrieve(url, file_path)
    print(f"Downloaded and saved to '{file_path}'.")

In [2]:
# Read the text file.
with open(file=file_path, mode="r", encoding="utf-8") as f:
    raw_text = f.read()

# Print the first 1000 characters of the text.
n_chars = 100
print(f"Total number of characters in '{file_path}': {len(raw_text)}\n")
print(f"First {n_chars} characters of '{file_path}':\n\n{raw_text[:n_chars]}")

Total number of characters in 'the_verdict/the-verdict.txt': 20479

First 100 characters of 'the_verdict/the-verdict.txt':

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [3]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(f"General regular expression result:\n     {result}\n")

# Modify the regular expression to split on whitespaces (\s), commas, and periods ([,.]).
result = re.split(r'([,.]|\s)', text)
print(f"Modified regular expression result:\n     {result}")

# Remove empty strings and whitespace-only strings from the result.
result = [item for item in result if item.strip()]
print(f"Final result after removing empty strings and whitespace-only strings:\n     {result}")

General regular expression result:
     ['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']

Modified regular expression result:
     ['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']
Final result after removing empty strings and whitespace-only strings:
     ['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


<div class="alert alert-block alert-info">

__Difference between `pattern=r'(\s)'` and `pattern=r'\s+'` when used with `re.split`:__

__`pattern=r'\s+'`__
- __Meaning__: Matches one or more consecutive whitespace characters (spaces, tabs, newlines, etc.).
- __Behavior in__ `re.split`: Splits the string at every sequence of whitespace, and does not include the whitespace in the result.
- __Example__:
    ```python
    import re
    text = "Hello   world!\nHow are you?"
    print(re.split(r'\s+', text))
    # Output: ['Hello', 'world!', 'How', 'are', 'you?']
    ```

__`pattern=r'(\s)'`__
- __Meaning__: Matches a single whitespace character, and the parentheses create a capturing group.
- __Behavior in__ `re.split`: Splits the string at every single whitespace character, and includes the matched whitespace characters in the result as separate elements.
- __Example__:
    ```python
    import re
    text = "Hello   world!\nHow are you?"
    print(re.split(r'(\s)', text))
    # Output: ['Hello', ' ', '', ' ', '', ' ', 'world!', '\n', 'How', ' ', 'are', ' ', 'you?']
    ```

__Summary Table:__

| Pattern   | Splits on              | Includes whitespace in result? | Example Output                                      |
|-----------|------------------------|--------------------------------|-----------------------------------------------------|
| `r'\s+'`  | Any run of whitespace  | No                             | `['Hello', 'world!', 'How', 'are', 'you?']`         |
| `r'(\s)'` | Each whitespace char   | Yes (as separate elements)     | `['Hello', ' ', '', ' ', '', ' ', ...]`             |

__In short:__
- Use `r'\s+'` to split and discard whitespace.
- Use `r'(\s)'` to split and keep each whitespace character in the result.

</div>

<div>
<img src="_docs/22-tokenizing-text-02.png" width="800"/>
</div>

In [4]:
# Modify the regular expression to also split on other punctuation marks such as colons, semicolons, question marks, exclamation marks, parentheses, and quotation marks.
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(f"Final result after removing empty strings and whitespace-only strings:\n     {result}")

Final result after removing empty strings and whitespace-only strings:
     ['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [5]:
# Preprocess the raw text by splitting on whitespaces and punctuation marks.
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"Total number of tokens after preprocessing: {len(preprocessed)}\n")

# Print the first 30 tokens after preprocessing.
print(f"First 30 tokens after preprocessing:\n    {preprocessed[:30]}")

Total number of tokens after preprocessing: 4690

First 30 tokens after preprocessing:
    ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


##### _2.3 Converting tokens into token IDs_

<div>
<img src="_docs/23-converting-tokens-into-tokenids-01.png" width="800"/>
</div>

In [6]:
# Create a sorted list of unique tokens and calculate the vocabulary size.
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocabulary size: {vocab_size}\n")

# Create a vocabulary.
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

Vocabulary size: 1130

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


<div class="alert alert-block alert-info">

```python
# Create a sample of text.
n_chars = 1000
n_tokens = 40
text_sample = raw_text[:n_chars]
print(f"Sample of {n_chars} characters from '{file_path}':\n\n{text_sample}\n")

# Split at 'white-space' (\s) characters (excluding).
pattern = r"\s+"
# Split the text into tokens.
text_tokens = re.split(pattern=pattern, string=text_sample)
print(f"Pattern: r'{pattern}'")
print(f"Number of tokens in the sample: {len(text_tokens)}")
print(
    f"First {n_tokens} tokens in the sample:\n{text_tokens[:n_tokens]}\n"
)  # Print the first 10 tokens.

# Split at 'white-space' character (including).
pattern = r"(\s)"
text_tokens = re.split(pattern=pattern, string=text_sample)
print(f"Pattern: r'{pattern}'")
print(f"Number of tokens in the sample: {len(text_tokens)}")
print(f"First {n_tokens} tokens in the sample:\n{text_tokens[:n_tokens]}\n")

# Split at 'white-space' characters (\s) and commas and period [,.] (excluding)
pattern = r"\s+|[,.]"
text_tokens = re.split(pattern=pattern, string=text_sample)
print(f"Pattern: r'{pattern}'")
print(f"Number of tokens in the sample: {len(text_tokens)}")
print(f"First {n_tokens} tokens in the sample:\n{text_tokens[:n_tokens]}\n")

# Split at 'white-space' character (\s) and commas and period [,.] (including).
pattern = r"(\s+|[,.])"
text_tokens = re.split(pattern=pattern, string=text_sample)
print(f"Pattern: r'{pattern}'")
print(f"Number of tokens in the sample: {len(text_tokens)}")
print(f"First {n_tokens} tokens in the sample:\n{text_tokens[:n_tokens]}\n")
```

__`pattern = r'\s+|([,.:;?_!"()\'-]|--)'`__

When the above pattern is used, `None` values are generated in the split result because the regex pattern uses a capturing group.

When a capturing group is used in `re.split`, the matched text for the group is included in the result. If a split occurs on the part that matches `\s+` (whitespace), the capturing group does not match anything. Hence, `re.split` inserts `None` in the result for that split.

__How to Fix?__
After splitting, filter out `None` values from your token list.

```python
import re

text = "Hello   world! How are you? -- I'm fine."
pattern = r'\s+|([,.:;?_!\"()\'-]|--)'
tokens = re.split(pattern, text)
# Remove None and empty strings
tokens = [tok for tok in tokens if tok not in (None, '')]
print(tokens)
# Output: ['Hello', 'world', '!', 'How', 'are', 'you', '?', '--', "I'm", 'fine', '.']
```

</div>

<div class="alert alert-block alert-info">

Explanation for 
```python
text = re.sub(r'\s+([,.:;?_!"()\'-]|--)', r'\1', text)
```

- Pattern: `r'\s+([,.:;?_!"()\'-]|--)'`

    - `\s+` matches one or more whitespace characters (spaces, tabs, newlines, etc.).
    - `([,.:;?_!"()\'-]|--)` is a capturing group that matches any one of the listed punctuation marks or a double dash (--).

- Replacement: `r'\1'`
    - `\1` refers to the text matched by the capturing group (the punctuation).

- What it does:
    - __Finds__: Any whitespace that comes immediately before one of the listed punctuation marks.
    - __Replaces__: The whitespace and the punctuation with just the punctuation (removes the whitespace before punctuation).

- Example:
    ```python
    text = "Hello , world ! How are you ? -- I'm fine ."
    text = re.sub(r'\s+([,.:;?_!"()\'-]|--)', r'\1', text)
    print(text)
    # Hello, world! How are you?-- I'm fine.
    ```

- Summary:

    This line removes any whitespace that appears before punctuation, so punctuation is directly attached to the preceding word. This is useful for detokenizing text after splitting punctuation into separate tokens.

</div>

<div>
<img src="_docs/23-converting-tokens-into-tokenids-02.png" width="800"/>
</div>

In [7]:
# Implement a simple text tokenizer.
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # Store the vocabulary as a class attribute for access in the encode and decode methods.
        self.str_to_int = vocab
        # Create an inverse vocabulary that maps token IDs back to the original text tokens.
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        # Process input text into token IDs.
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        # Convert token IDs back into text.
        text = " ".join([self.int_to_str[i] for i in ids]) 

        # Remove spaces before the specified punctuation.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

<div>
<img src="_docs/23-converting-tokens-into-tokenids-03.png" width="800"/><br><br>
</div>

In [8]:
# Test the tokenizer with a sample text.
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(f"Encoded IDs:\n     {ids}")

decoded_text = tokenizer.decode(ids)
print(f"Decoded text:\n     {decoded_text}")

Encoded IDs:
     [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
Decoded text:
     " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


<div class="alert alert-block alert-warning">

```python
# Test the tokenizer with another sample text, not contained in training set.
text = "Hello, do you like tea?"
print(tokenizer.encode(text))
```

</div>

##### _2.4 Adding special context tokens_

<div>
<img src="_docs/24-adding-special-context-token-01.png" width="800"/><br><br>
<img src="_docs/24-adding-special-context-token-02.png" width="800"/>
</div>

In [9]:
# Modify the vocabulary to include two special tokens, <unk> and <|endoftext|>.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

print(f"Vocabulary size: {len(vocab.items())}\n")

# Print last five entries of the updated vocabulary.
print("Last five entries of the updated vocabulary:")
for i, item in enumerate(list(vocab.items())[-5:]):
    print(f"     {item}")

Vocabulary size: 1132

Last five entries of the updated vocabulary:
     ('younger', 1127)
     ('your', 1128)
     ('yourself', 1129)
     ('<|endoftext|>', 1130)
     ('<|unk|>', 1131)


In [10]:
# Implement a simple text tokenizer that handles unknown words.
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Replace unknown words by <|unk|> tokens.
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        # Replace spaces before the specified punctuations.
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [11]:
# Test the tokenizer with a sample text that includes unknown words.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(f"Sample text is:\n     {text}\n")

# Encode the sample text using the modified tokenizer that handles unknown words.
tokenizer = SimpleTokenizerV2(vocab)
print(f"Encoded text is:\n     {tokenizer.encode(text)}\n")

# Decode the encoded text back to the original text.
print(f"Decoded text is:\n     {tokenizer.decode(tokenizer.encode(text))}")

Sample text is:
     Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

Encoded text is:
     [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

Decoded text is:
     <|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


<div class="alert alert-block alert-info">

Depending on the LLM, some researchers also consider additional special tokens such as the following:
- [`BOS`] (beginning of sequence) —This token marks the start of a text. It signifies to the LLM where a piece of content begins.  
- [`EOS`] (end of sequence) —This token is positioned at the end of a text and is especially useful when concatenating multiple unrelated texts, similar to <|endoftext|>. For instance, when combining two different Wikipedia articles or books, the [`EOS`] token indicates where one ends and the next begins.  
- [`PAD`] (padding) —When training LLMs with batch sizes larger than one, the batch might contain texts of varying lengths. To ensure all texts have the same length, the shorter texts are extended or “padded” using the [`PAD`] token, up to the length of the longest text in the batch.

</div>

##### _2.5 Byte pair encoding (BPE)_

<div class="alert alert-block alert-info">
Check <a href="https://github.com/openai/tiktoken">tiktoken</a> for more details on BPE.

```python
?tiktoken.get_encoding
# help(tiktoken.get_encoding)
```

```
Signature: tiktoken.get_encoding(encoding_name: 'str') -> 'Encoding'
Docstring: <no docstring>
File:      ~/Study/github/python-examples/llm-from-scratch/.venv/lib/python3.11/site-packages/tiktoken/registry.py
Type:      function
```

</div>

<div>
<img src="_docs/25-byte pair-encoding.png" width="800"/>
</div>

In [12]:
# uv add tiktoken
from importlib.metadata import version
import tiktoken
print(f"tiktoken version: {version('tiktoken')}")

tiktoken version: 0.9.0


In [13]:
# Create a BPE tokenizer instance from 'tiktoken'.
tokenizer = tiktoken.get_encoding(encoding_name="gpt2")

# Encode a sample text using the 'tiktoken' tokenizer, allowing the special token <|endoftext|>.
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(f"Encoded integers:\n     {integers}\n")

# Decode the encoded integers back to the original text using the 'tiktoken' tokenizer.
strings = tokenizer.decode(integers)
print(f"Decoded text:\n     {strings}\n")


Encoded integers:
     [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]

Decoded text:
     Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.



<div class="alert alert-block alert-info">

```python
# Define a sample text.
text_sample = """
Hello, world...!!! <|endoftext|> Welcome to the world of LLMs.
"""

# Encode the text.
text_encoded = tokenizer.encode(text=text_sample, allowed_special="all")
print(f"Encoded text:\n     {text_encoded}\n")

# Decode the text.
text_decoded = tokenizer.decode(tokens=text_encoded)
print(f"Decoded text:\n     {text_decoded}")

# Define random text.
text_sample = """
hsakhgkk 798796 ^%$jvja ":>>L)(*)
"""

# Encode the text.
text_encoded = tokenizer.encode(text=text_sample, allowed_special="all")
print(f"Encoded 'random' text:\n    {text_encoded}\n")

# Decode the text.
text_decoded = tokenizer.decode(tokens=text_encoded)
print(f"Decoded 'random' text:\n     {text_decoded}")
```

</div>

##### _2.6 Data sampling with a sliding window_

<div>
<img src="_docs/26-data-sampling-with-sliding-window-01.png" width="800"/>
</div>

In [14]:
file_path

'the_verdict/the-verdict.txt'

In [15]:
# Read and encode the raw text from "the-verdict.txt" using the 'tiktoken' tokenizer.
with open(file=file_path, mode="r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(f"Length of encoded text: {len(enc_text)}\n")

# Remove the first 50 tokens from the encoded text (dataset).
enc_sample = enc_text[50:]

# Context size determines how many tokens are included in the input.
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x:\n     {x}")
print(f"y:\n     {y}\n") 

print(f"{'==' * 50}\n")
for i in range(1, context_size + 1):
    x = enc_sample[:i]
    y = enc_sample[i]
    print(f"{x} ---> {y}\n")

print(f"{'==' * 50}\n")
for i in range(1, context_size + 1):
    x = tokenizer.decode(tokens=enc_sample[:i])
    y = tokenizer.decode(tokens=[enc_sample[i]])
    print(f"{x} ---> {y}")
     

Length of encoded text: 5145

x:
     [290, 4920, 2241, 287]
y:
     [4920, 2241, 287, 257]


[290] ---> 4920

[290, 4920] ---> 2241

[290, 4920, 2241] ---> 287

[290, 4920, 2241, 287] ---> 257


 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


<div>
<img src="_docs/26-data-sampling-with-sliding-window-02.png" width="800"/>
</div>

In [16]:
# uv add torch torchvision
import torch
from torch.utils.data import Dataset, DataLoader

# Dataset class for batched input and target.
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text.
        token_ids = tokenizer.encode(text=text)

        # [i for i in range(0, (45 - 4), 4)]
        # [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40]

        # Use a sliding window to chunk the book into overlapping sequences of max_length.
        for i in range(0, len(token_ids) - context_length, stride):
            input_chunk = token_ids[i : i + context_length]
            target_chunk = token_ids[i + 1 : i + context_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # Return the total number of rows in the dataset.
    def __len__(self):
        return len(self.input_ids)

    # Return a single row from the dataset.
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [17]:
# Function to create a dataloader that generates batches of input and target sequences.
def create_dataloader_v1(
    text,
    batch_size=4,
    context_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    # Initialize the tokenizer.
    tokenizer = tiktoken.get_encoding(encoding_name="gpt2")

    # Create dataset.
    dataset = GPTDatasetV1(
        text=text, tokenizer=tokenizer, context_length=context_length, stride=stride
    )

    # Create dataloader.
    dataloader = DataLoader(
        dataset=dataset,  # dataset from which to load the data.
        batch_size=batch_size,  # number of samples per batch to load.
        shuffle=shuffle,
        num_workers=num_workers,  # number of subprocesses (CPU processes) to use for data preprocessing.
        drop_last=drop_last,  # drop_last=True drops the last batch, if it is shorter than the specified batch_size to prevent loss spikes during training.
    )

    return dataloader

In [18]:
# Check PyTorch version.
print(f"PyTorch version: {torch.__version__}\n")
print(f"{'==' * 50}\n")

# Read the text/book.
with open(file=file_path, mode="r", encoding="utf-8") as f:
    text_raw = f.read()

# Load the text into a DataLoader.
dataloader = create_dataloader_v1(
    text=text_raw,
    batch_size=1,
    context_length=4,
    stride=1,
    shuffle=False,
    num_workers=0,
)

# Convert dataloader into a Python iterator to fetch the next entry via Python’s built-in next() function.
data_iter = iter(dataloader)
# list(data_iter)

# Fetch the first entry from the dataloader.
x, y = next(data_iter)

# Print the first batch.
print(f"First batch of input ids (x):\n     {x}\n")
print(f"First batch of target ids (y):\n     {y}\n")

print(f"{'==' * 50}\n")

# Fetch the second entry from the dataloader.
x, y = next(data_iter)

# Print the second batch.
print(f"Second batch of input ids (x):\n     {x}\n")
print(f"Second batch of target ids (y):\n     {y}")

# print(next(data_iter))

PyTorch version: 2.7.0


First batch of input ids (x):
     tensor([[  40,  367, 2885, 1464]])

First batch of target ids (y):
     tensor([[ 367, 2885, 1464, 1807]])


Second batch of input ids (x):
     tensor([[ 367, 2885, 1464, 1807]])

Second batch of target ids (y):
     tensor([[2885, 1464, 1807, 3619]])


<div>
<img src="_docs/26-data-sampling-with-sliding-window-03.png" width="800"/>
</div>

In [19]:
# Define batch size.
batch_size, stride = 8, 1
print(f"{batch_size = }, {stride = }\n")
print(f"{'==' * 50}\n")

# Load the text into a DataLoader with the new batch size and stride.
dataloader = create_dataloader_v1(
    text=text_raw,
    batch_size=batch_size,
    context_length=4,
    stride=stride,
    shuffle=False,
    num_workers=0,
)
data_iter = iter(dataloader)

# Print the first batch.
x, y = next(data_iter)
print(f"First batch of input ids (x):\n     {x}\n")
print(f"First batch of target ids (y):\n     {y}\n")

print(f"{'==' * 50}\n")

# Print the second batch.
x, y = next(data_iter)
print(f"Second batch of input ids (x):\n     {x}\n")
print(f"Second batch of target ids (y):\n     {y}")

batch_size = 8, stride = 1


First batch of input ids (x):
     tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]])

First batch of target ids (y):
     tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026]])


Second batch of input ids (x):
     tensor([[10899,  2138,   257,  7026],
        [ 2138,   257,  7026, 15632],
        [  257,  7026, 15632,   438],
        [ 7026, 15632,   438,  2016],
        [15632,   438,  2016,   257],
        [  438,  2016,   257,   922],
        [ 2016,   257, 

##### _2.7 Creating token embeddings_

<div>
<img src="_docs/27-creating-token-embeddings-01.png" width="800"/>
</div>


In [20]:
torch.manual_seed(123)

# Example input IDs for testing the embedding layer.
input_ids = torch.tensor([2, 3, 5, 1])  

# Define the vocabulary size and output dimension for the embedding layer.
vocab_size = 6
output_dim = 3  

# Create an embedding layer and print its weights.
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(f"Weight of embedding layer -->\n{embedding_layer.weight}\n")

print(f"{'==' * 50}\n")

# Test the embedding layer by passing an example input ID and printing the resulting embedding vector.
print(f"Embedding vector for input ID 3 -->\n{embedding_layer(torch.tensor([3]))}\n")

print(f"{'==' * 50}\n")

# Pass the batch of input IDs through the embedding layer and print the resulting embedding vectors.
print(f"Embedding vectors for input IDs {input_ids.tolist()} -->\n{embedding_layer(input_ids)}")


Weight of embedding layer -->
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


Embedding vector for input ID 3 -->
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


Embedding vectors for input IDs [2, 3, 5, 1] -->
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


<div>
<img src="_docs/27-creating-token-embeddings-02.png" width="800"/>
</div>

##### _2.8 Encoding word positions_

<div>
<img src="_docs/28-encoding-word-positions-01.png" width="800"/><br><br>
<img src="_docs/28-encoding-word-positions-02.png" width="800"/>
</div>

In [33]:
# Define the vocabulary size, output dimension, and context length for the embedding layer and dataloader.
vocab_size = 50257
output_dim = 256
context_length = 4
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Load the text into a DataLoader with the specified batch size and stride.
dataloader = create_dataloader_v1(
    text=raw_text, 
    batch_size=8, 
    context_length=context_length,
    stride=context_length, 
    shuffle=False
)

# Get one batch of input and target IDs from the dataloader and print their shapes.
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print(f"Token IDs for one input batch:\n{inputs}\n")
print(f"Inputs shape: {inputs.shape}\n")

print(f"{'==' * 50}\n")

print(f"Token IDs for one target batch:\n{targets}\n")
print(f"Targets shape: {targets.shape}\n")

print(f"{'==' * 50}\n")

token_embeddings = token_embedding_layer(inputs)
print(f"Token embeddings shape for one batch: {token_embeddings.shape}\n")

print(f"{'==' * 50}\n")

pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(f"Positional embeddings shape: {pos_embeddings.shape}\n")

print(f"{'==' * 50}\n")

input_embeddings = token_embeddings + pos_embeddings
print(f"Input embeddings shape for one batch: {input_embeddings.shape}")

Token IDs for one input batch:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape: torch.Size([8, 4])


Token IDs for one target batch:
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

Targets shape: torch.Size([8, 4])


Token embeddings shape for one batch: torch.Size([8, 4, 256])


Positional embeddings shape: torch.Size([4, 256])


Input embeddings shape for one batch: torch.Size([8, 4, 256])


<div>
<img src="_docs/28-encoding-word-positions-03.png" width="800"/>
</div>


#### 3 Coding attention mechanisms

<div>
<img src="_docs/30-coding-attention-mechanisms-01.png" width="800"/><br><br>
<img src="_docs/30-coding-attention-mechanisms-02.png" width="800"/>
</div>

##### _3.1 The problem with modeling long sequences_

<div>
<img src="_docs/31-problem-modeling-long-sequences-01.png" width="800"/><br><br>
<img src="_docs/31-problem-modeling-long-sequences-02.png" width="800"/>
</div>

##### _3.2 Capturing data dependencies with attention mechanisms_

<div>
<img src="_docs/32-capturing-data-dependencies-with-attention-mechanisms-01.png" width="800"/><br><br>
<img src="_docs/32-capturing-data-dependencies-with-attention-mechanisms-02.png" width="800"/>
</div>

##### _3.3 Attending to different parts of the input with self-attention_

_3.3.1 A simple self-attention mechanism without trainable weights_

<div>
<img src="_docs/33-1-simple-self-attention-mechanism-without-trainable-weights-01.png" width="800"/>
</div>

<div class="alert alert-block alert-warning">
Warning message here...
</div>

Token/Vector embedding.

In [ ]:
# uv add gensim
# from gensim.models import Word2Vec
import gensim.downloader as api

model = api.load(name="word2vec-google-news-300", return_path=True)
print(f"Model path: {model}\n")

print(help(api.load))

In [ ]:
model = api.load(name="word2vec-google-news-300", return_path=False)
print(f"Model type: {type(model)}")

In [ ]:
# Print the token/vector embedding for the word 'cricket'.
print(f"Token/Vector embedding for the word 'cricket':\n{model['cricket']}\n")
print(f"{'==' * 50}\n")

# Print the shape of token/vector embedding for the word 'cricket'.
print(
    f"Shape of token/vector embedding for the word 'cricket': {model['cricket'].shape}\n"
)
print(f"{'==' * 50}\n")

# Print the token/vector embedding for the word 'cricket' as a list.
print(
    f"Token/vector embedding for the word 'cricket' as a list:\n{model['cricket'].tolist()}"
)

In [ ]:
# help(model.most_similar)

# Example of 'most_similar' method.
print(f"Most similar words to 'cricket':\n{model.most_similar('cricket')}\n")
print(f"{'==' * 50}\n")

# Example of 'most_similar' method with 'topn' parameter.
print(
    f"Most similar words to 'cricket' with 'topn=5':\n{model.most_similar('cricket', topn=5)}\n"
)
print(f"{'==' * 50}\n")

print(
    f"Most similar words for 'King + Woman - Man' with 'topn=10':\n{model.most_similar(positive=['King', 'Woman'], negative=['Man'], topn=10)}\n"
)
print(f"{'==' * 50}\n")

print(
    f"Most similar words for 'king + woman - man' with 'topn=10':\n{model.most_similar(positive=['king', 'woman'], negative=['man'], topn=10)}"
)

In [ ]:
# help(model.similarity)

# Example of 'similarity' method.
print(
    f"Similarity between 'king' and 'queen': {model.similarity(w1='king', w2='queen')}"
)
print(f"Similarity between 'man' and 'woman': {model.similarity(w1='man', w2='woman')}")
print(
    f"Similarity between 'earth' and 'planet': {model.similarity(w1='earth', w2='planet')}"
)
print(
    f"Similarity between 'cricket' and 'football': {model.similarity(w1='cricket', w2='football')}\n"
)

print(f"{'==' * 50}\n")

print(
    f"Similarity between 'paper' and 'water': {model.similarity(w1='paper', w2='water')}"
)
print(
    f"Similarity between 'house' and 'food': {model.similarity(w1='house', w2='food')}"
)
print(f"Similarity between 'car' and 'pen': {model.similarity(w1='car', w2='pen')}")
print(
    f"Similarity between 'computer' and 'sword': {model.similarity(w1='computer', w2='sword')}"
)

In [ ]:
import numpy as np


# Calculate the vector difference between two words.
def vector_difference(model, word1, word2):
    return model[word1] - model[word2]


vec_diff_man_woman = vector_difference(model=model, word1="man", word2="woman")
vec_diff_computer_sword = vector_difference(
    model=model, word1="computer", word2="sword"
)

# print(f"Vector difference between 'man' and 'woman':\n{vec_diff_man_woman}\n")
# print(f"Vector difference between 'king' and 'queen':\n{vec_diff_king_queen}\n")

print(
    f"Magnitude of vector difference between 'man' and 'woman': {np.linalg.norm(vec_diff_man_woman)}"
)
print(
    f"Magnitude of vector difference between 'computer' and 'sword': {np.linalg.norm(vec_diff_computer_sword)}"
)

Create token/vector embeddings

In [ ]:
# Example of input token ids.
input_ids = torch.tensor([2, 3, 5, 6])

# Define the vocabulary size and embedding dimension.
vocab_size = 10
embedding_dim = 4

torch.manual_seed(42)  # Set seed for reproducibility.

# Create a random embedding matrix.
embedding_layer = torch.nn.Embedding(
    num_embeddings=vocab_size, embedding_dim=embedding_dim
)

print(f"Embedding layer weights:\n{embedding_layer.weight}\n")
print(f"Token/Vector embeddings for 3rd token id:\n{embedding_layer.weight[2]}\n")

# Get the token embeddings for the input ids.
token_embeddings = embedding_layer(input_ids)
print(f"Token embeddings for input ids {input_ids.tolist()}:\n{token_embeddings}")

Positional embeddings (encoding word positions)

In [ ]:
# Embedding dimension for token and positional embeddings.
# token_embedding_dim = pos_embedding_dim
embedding_dim = 256
vocab_size = 50257
context_size = 4

# Create a token embedding layer.
token_embedding_layer = torch.nn.Embedding(
    num_embeddings=vocab_size, embedding_dim=embedding_dim
)

text_encoded = tokenizer.encode(text=text_raw, allowed_special="all")
print(f"Sample of encoded text:\n{text_encoded[:100]}")

In [ ]:
dataloader = create_dataloader_v1(
    text=text_raw,
    batch_size=8,
    context_length=context_size,
    stride=context_size,
    shuffle=False,
    num_workers=0,
)

data_iter = iter(dataloader)

input_ids, target_ids = next(data_iter)
print(f"Input ids:\n{input_ids}\n")
print(f"Target ids:\n{target_ids}\n")

print(f"Input ids' shape: {input_ids.shape}")
print(f"Target ids' shape: {target_ids.shape}\n")

token_embeddings = token_embedding_layer(input_ids)
print(f"Token embeddings' shape: {token_embeddings.shape}")

# Create a positional embedding layer.
pos_embedding_layer = torch.nn.Embedding(
    num_embeddings=context_size, embedding_dim=embedding_dim
)
pos_embeddings = pos_embedding_layer(torch.arange(context_size))
print(f"Positional token embeddings' shape: {pos_embeddings.shape}")

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(f"Input embeddings' shape: {input_embeddings.shape}")

- #### Step #2 - Attention mechanism

Simplified self attention

In [ ]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with (x^4)
        [0.77, 0.25, 0.10],  # one (x^5)
        [0.05, 0.80, 0.55],
    ]  # step (x^6)
)

In [ ]:
import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D

# Corresponding words.
words = ["Your", "journey", "starts", "with", "one", "step"]

# Extract x, y, z coordinates.
x_coords = inputs[:, 0].numpy()
y_coords = inputs[:, 1].numpy()
z_coords = inputs[:, 2].numpy()

# Create a 3D scatter plot.
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(111, projection="3d")

# Plot each point and annotate with the corresponding word.
for x, y, z, word in zip(x_coords, y_coords, z_coords, words):
    ax.scatter(x, y, z)
    ax.text(x, y, z, word, fontsize=12, ha="right")

# Set labels for axes.
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.set_zlabel("Z-axis")

plt.title("3D Scatter Plot of Word Embeddings")
plt.tight_layout()
plt.show()

In [ ]:
# Create 3D scatter plot with vectors from origin to each point, using different colors.
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(111, projection="3d")

# Define colors for each point.
colors = ["r", "g", "b", "c", "m", "y"]

# Plot each vector with a different color and annotate with the corresponding word.
for x, y, z, word, color in zip(x_coords, y_coords, z_coords, words, colors):
    ax.quiver(0, 0, 0, x, y, z, color=color, arrow_length_ratio=0.03)
    ax.text(x, y, z, word, fontsize=12, ha="right", color=color)

# Set labels for axes.
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.set_zlabel("Z-axis")

# Set the limits for each axis.
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_zlim([0, 1])

plt.title("3D Scatter Plot of Word Embeddings with Vectors")
plt.tight_layout()
plt.show()

In [ ]:
# 2nd example of inputs.
query = inputs[1]
print(f"Input vector: \n{inputs}\n")
print(f"Query/Input vector (x^2): {query}\n")

print(f"Shape of inputs: {inputs.shape}")
print(f"Shape of query: {query.shape}\n")

# Calculate attention scores using dot product.
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(f"{attn_scores_2 = }")
print(f"Shape of attention scores: {attn_scores_2.shape}\n")

# Summation normalisation.
attn_weights_2_sum_norm = attn_scores_2 / attn_scores_2.sum()
print(f"{attn_weights_2_sum_norm = }")
print(f"Sum of attention weights: {attn_weights_2_sum_norm.sum()}\n")


# Naive softmax normalisation.
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)


attn_weights_2_naive_softmax_norm = softmax_naive(attn_scores_2)
print(f"{attn_weights_2_naive_softmax_norm = }")
print(f"Sum of attention weights: {attn_weights_2_naive_softmax_norm.sum()}\n")

# Pytorch softmax normalisation.
attn_weights_2_torch_softmax_norm = torch.softmax(attn_scores_2, dim=0)
print(f"{attn_weights_2_torch_softmax_norm = }")
print(f"Sum of attention weights: {attn_weights_2_torch_softmax_norm.sum()}\n")

# Context vector calculation.
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2_torch_softmax_norm[i] * x_i
print(f"{context_vec_2 = }")
print(f"Shape of context vector: {context_vec_2.shape}")

In [ ]:
# Calculate attention scores using dot product, for entire input.
# attn_scores = torch.empty(6, 6)
attn_scores = torch.empty(inputs.shape[0], inputs.shape[0])
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(f"Attention score (normal 'for-loop' multiplication):\n{attn_scores}\n")

# Calculate attention scores using matrix multiplication.
attn_scores = inputs @ inputs.T
print(f"Attention score ('matrix' multiplication):\n{attn_scores}\n")

# Normalise attention scores using softmax.
attn_weights = torch.softmax(attn_scores, dim=-1)
print(f"Attention weights (normalised):\n{attn_weights}\n")

row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print(f"{row_2_sum = }")
print(f"All row sums: {attn_weights.sum(dim=-1)}")

In [ ]:
print(f"Shape of attention scores: {attn_scores.shape}")
print(f"Shape of attention weights: {attn_weights.shape}")
print(f"Shape of inputs: {inputs.shape}\n")

all_context_vecs = attn_weights @ inputs
print(f"Context vectore:\n{all_context_vecs}\n")
print(f"Shape of context vectors: {all_context_vecs.shape}\n")
print(f"Previous 2nd context vector: {context_vec_2}")

In [ ]:
# Different normalisation methods, extreme values.
import math

x = [1, 2, 3, 400]

x_sum_norm = [xi / sum(x) for xi in x]
print(f"{x_sum_norm = }\n")

x_exp_norm = [math.exp(xi) / sum(math.exp(xi) for xi in x) for xi in x]
print(f"{x_exp_norm = }\n")

x_torch_softmax = torch.softmax(torch.tensor(x, dtype=torch.float32), dim=0)
print(f"{x_torch_softmax = }")

What is the difference between the following?

- `torch.softmax(attn_scores, dim=0)`
    - Softmax is applied down each column (along rows).
    - For each column, the values in that column are exponentiated and normalized so the column sums to 1.
    - Each column sums to 1.

- `torch.softmax(attn_scores, dim=1)`
    - Softmax is applied across each row (along columns).
    - For each row, the values in that row are exponentiated and normalized so the row sums to 1.
    - Each row sums to 1.
    - This is the standard for attention mechanisms.

- `torch.softmax(attn_scores, dim=-1)`
    - `dim=-1` means the last dimension (for 2D, same as `dim=1`).
    - Softmax is applied across each row (just like `dim=1` for 2D tensors).
    - Each row sums to 1.

Summary Table:

| Call                            | Softmax Applied On   | Sums to 1?         | Typical for Attention? |
|---------------------------------|----------------------|--------------------|------------------------|
| `torch.softmax(..., dim=0)`     | Down columns         | Each column        | No                     |
| `torch.softmax(..., dim=1)`     | Across rows          | Each row           | Yes                    |
| `torch.softmax(..., dim=-1)`    | Across last axis     | Each row (2D)      | Yes                    |

In attention, you almost always want `dim=1` or `dim=-1` so each row (query) gets a probability distribution over keys.



Self attention (with trainable weights)

In [ ]:
# in GPT-like models, the input and output dimensions are usually the same.
x_2 = inputs[1]  # The second input element.
d_in = inputs.shape[1]  # The input embedding size, d=3.
d_out = 2  # The output embedding size, d_out=2.

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

keys = inputs @ W_key
values = inputs @ W_value
print("\nkeys.shape:", keys.shape)
print("values.shape:", values.shape)

In [ ]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

In [ ]:
# Why divide by sqrt(d_k)?
# The division by sqrt(d_k) is a scaling factor to prevent the dot product from growing too large, which can lead to numerical instability and saturation of the softmax function.
# This scaling helps to keep the gradients in a reasonable range during training, making the model more stable and easier to optimize.

# Define a tensor.
tensor_x = torch.tensor(data=[0.1, -0.2, 0.3, -0.4, 0.5], dtype=torch.float32)

# Apply softmax without scaling.
softmax_x = torch.softmax(tensor_x, dim=-1)
print(f"Softmax without scaling:\n{softmax_x}\n")

# Multiply the tensor by a large number.
large_number = 10
tensor_x_large = tensor_x * large_number
# Apply softmax without scaling.
softmax_x_large = torch.softmax(tensor_x_large, dim=-1)
print(f"Softmax without scaling (large tensor):\n{softmax_x_large}\n")

# Apply softmax with scaling.
scaled_tensor_x = tensor_x / (5**0.5)  # dim = 5, 1 row and 5 columns.
softmax_x_scaled = torch.softmax(scaled_tensor_x, dim=-1)
print(f"Softmax with scaling:\n{softmax_x_scaled}")

# # Apply softmax with scaling on the large tensor.
# scaled_tensor_x_large = tensor_x_large / (5 ** 0.5)
# softmax_x_scaled_large = torch.softmax(scaled_tensor_x_large, dim=-1)
# print(f"Softmax with scaling (large tensor):\n{softmax_x_scaled_large}\n")

In [ ]:
import numpy as np


# Function to compute variance before and after scaling.
def compute_variance(dim, num_trails=1000):
    dot_products = []
    dot_products_scaled = []

    # Set random seed for reproducibility.
    np.random.seed(14)

    # Generate multiple random vectors and compute dot products.
    for _ in range(num_trails):
        # Generate random vectors.
        _query = np.random.randn(dim)
        _key = np.random.randn(dim)

        # Compute dot product.
        dot_products.append(np.dot(_query, _key))

        # Compute scaled dot product.
        scaled_dot_product = np.dot(_query, _key) / np.sqrt(dim)
        dot_products_scaled.append(scaled_dot_product)

    #  Calculate variance of dot products.
    variance_before_scaling = np.var(dot_products)
    variance_after_scaling = np.var(dot_products_scaled)

    return variance_before_scaling, variance_after_scaling


# Compute variance for different dimensions.
dim = 10
variance_before, variance_after = compute_variance(dim=dim)
print(f"Variance before scaling (dim={dim}): {variance_before}")
print(f"Variance after scaling (dim={dim}): {variance_after}\n")

dim = 100
variance_before, variance_after = compute_variance(dim=dim)
print(f"Variance before scaling (dim={dim}): {variance_before}")
print(f"Variance after scaling (dim={dim}): {variance_after}\n")

dim = 1000
variance_before, variance_after = compute_variance(dim=dim)
print(f"Variance before scaling (dim={dim}): {variance_before}")
print(f"Variance after scaling (dim={dim}): {variance_after}\n")

dim = 10000
variance_before, variance_after = compute_variance(dim=dim)
print(f"Variance before scaling (dim={dim}): {variance_before}")
print(f"Variance after scaling (dim={dim}): {variance_after}")

<div>
<img src="_docs/self-attention-full-flow.png" width="800"/><br>
<img src="_docs/self-attention-step1.png" width="800"/><br>
<img src="_docs/self-attention-step2.png" width="800"/><br>
<img src="_docs/self-attention-step3.png" width="800"/><br>
<img src="_docs/self-attention-step4.png" width="800"/><br>
</div>

In [ ]:
import torch.nn as nn

# Set seed for reproducibility.
torch.manual_seed(14)


# Create a compact self-attention class.
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # Compute query, key, and value vectors.
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        # Compute attention scores.
        attn_scores = queries @ keys.T

        # Scale attention scores.
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

        # Compute context vectors.
        context_vecs = attn_weights @ values

        return context_vecs


# Create an instance of the self-attention class.
d_in = inputs.shape[1]  # Input embedding size.
d_out = 2  # Output embedding size.
sa_v1 = SelfAttention_v1(d_in=d_in, d_out=d_out)

print(f"'inputs':\n{inputs}\n")
print(f"SelfAttention_v1 for 'inputs':\n{sa_v1(inputs)}")

In [ ]:
# Set seed for reproducibility.
torch.manual_seed(14)


# Create a self-attention class with PyTorch's Linear layer.
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        # Compute query, key, and value vectors.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Compute attention scores.
        attn_scores = queries @ keys.T

        # Scale attention scores.
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

        # Compute context vectors.
        context_vecs = attn_weights @ values

        return context_vecs


# Create an instance of the self-attention class with PyTorch's Linear layer.
d_in = inputs.shape[1]  # Input embedding size.
d_out = 2  # Output embedding size.
sa_v2 = SelfAttention_v2(d_in=d_in, d_out=d_out)

print(f"'inputs':\n{inputs}\n")
print(f"SelfAttention_v2 for 'inputs':\n{sa_v2(inputs)}")

Causal attention

<div>
<img src="_docs/triangular-matrix.jpeg" width="800"/>
</div>

In [ ]:
print(f"'inputs':\n{inputs}\n")

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(f"Attention weights (normalised):\n {attn_weights}\n")

context_length = attn_scores.shape[0]
# tril - lower triangular matrix.
# triu - upper triangular matrix.
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(f"Simple mask to apply on attention weights:\n{mask_simple}")

In [ ]:
masked_simple = attn_weights * mask_simple
print(f"Attention weights after applying mask:\n{masked_simple}\n")

row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(f"Attention weights renormalised after masking:\n{masked_simple_norm}")

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print(f"Simple mask to apply on attention scores:\n{mask}\n")

masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(f"Attention scores masked and filled (replaced):\n{masked}\n")

attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=1)
print(f"Attention weights after masking and softmax:\n{attn_weights}\n")

print(f"Sum of each row in attention weights:\n{attn_weights.sum(dim=-1)}")

Why `triu` is used, instead of `tril`?

```python
print(f"attn_scores:\n{attn_scores}\n")
print(f"mask_simple:\n{mask_simple}\n")

attn_scores_masked = attn_scores * mask_simple
print(f"attn_scores * mask_simple:\n{attn_scores_masked}\n")

attn_weights_masked = torch.softmax(attn_scores_masked, dim=-1)
print(f"Attention weights after simple masking and softmax:\n{attn_weights_masked}\n")

print(f"Sum of each row in attention weights:\n{attn_weights.sum(dim=-1)}\n")
```
```
attn_scores:
tensor([[-0.0625, -0.3480, -0.3464, -0.2135, -0.2192, -0.2407],
        [ 0.2607,  0.0422,  0.0426, -0.0164,  0.0392, -0.0172],
        [ 0.2567,  0.0404,  0.0409, -0.0168,  0.0379, -0.0177],
        [ 0.1854,  0.0870,  0.0870,  0.0250,  0.0633,  0.0291],
        [ 0.1113, -0.0027, -0.0024, -0.0203,  0.0039, -0.0223],
        [ 0.2318,  0.1018,  0.1019,  0.0268,  0.0748,  0.0313]],
       grad_fn=<MmBackward0>)

mask_simple:
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

attn_scores * mask_simple:
tensor([[-0.0625, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000],
        [ 0.2607,  0.0422,  0.0000, -0.0000,  0.0000, -0.0000],
        [ 0.2567,  0.0404,  0.0409, -0.0000,  0.0000, -0.0000],
        [ 0.1854,  0.0870,  0.0870,  0.0250,  0.0000,  0.0000],
        [ 0.1113, -0.0027, -0.0024, -0.0203,  0.0039, -0.0000],
        [ 0.2318,  0.1018,  0.1019,  0.0268,  0.0748,  0.0313]],
       grad_fn=<MulBackward0>)

Attention weights after simple masking and softmax:
tensor([[0.1582, 0.1684, 0.1684, 0.1684, 0.1684, 0.1684],
        [0.2047, 0.1645, 0.1577, 0.1577, 0.1577, 0.1577],
        [0.2027, 0.1633, 0.1634, 0.1568, 0.1568, 0.1568],
        [0.1878, 0.1702, 0.1702, 0.1599, 0.1560, 0.1560],
        [0.1833, 0.1636, 0.1636, 0.1607, 0.1647, 0.1640],
        [0.1907, 0.1675, 0.1675, 0.1554, 0.1630, 0.1561]],
       grad_fn=<SoftmaxBackward0>)

Sum of each row in attention weights:
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)
```

In this approach, the values that we wanted to mask in the `attention_weights` matrix do carry information and is __NOT Zero__.

In [ ]:
torch.manual_seed(14)
dropout = torch.nn.Dropout(p=0.5)  # Set dropout probability to 50%.
example = torch.ones(6, 6)  # Example tensor to apply dropout on.
print(f"Example tensor before dropout:\n{example}\n")
print(f"Example tensor after dropout:\n{dropout(example)}")

The outputs are scaled by a factor of `1/(1-p)` during training. More details can be found [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html).

In [ ]:
torch.manual_seed(14)
print(f"Attention weights before dropout:\n{attn_weights}\n")
print(f"Attention weights after dropout:\n{dropout(attn_weights)}")

In [ ]:
# Create a batch of inputs by stacking two copies of the same input tensor.
batch = torch.stack((inputs, inputs), dim=0)
print(f"Batch of inputs:\n{batch}\n")
print(f"Shape of the batch: {batch.shape}\n")


# Implement a compact causal attention class.
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.context_length = context_length
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            name="mask",
            tensor=torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        b, num_tokens, d_in = (
            x.shape
        )  # Batch size, number of tokens, and input dimension.
        # Here, batch size (b) and input dimension (d_in) are not used, but they can be useful for further processing.

        # Compute query, key, and value vectors.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Compute attention scores.
        attn_scores = (
            queries @ keys.transpose(1, 2)
        )  # Transpose dimensions 1 and 2, keeping the batch dimension at the first position (0).

        # Mask attention scores using the pre-defined mask.
        # In PyTorch, operations with a trailing underscore are performed in-place, avoiding unnecessary memory copies.
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        # Normalise attention scores.
        attn_weights = torch.softmax(input=attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        # Apply dropout to attention weights.
        attn_weights = self.dropout(attn_weights)

        # Compute context vectors.
        context_vecs = attn_weights @ values

        return context_vecs

| Variable        | Shape                        | Meaning                                  |
|-----------------|-----------------------------|------------------------------------------|
| `batch`         | (b, num_tokens, d_in)       | Input batch of sequences                 |
| `queries`       | (b, num_tokens, d_out)      | Projected queries per batch              |
| `keys`          | (b, num_tokens, d_out)      | Projected keys per batch                 |
| `attn_scores`   | (b, num_tokens, num_tokens) | Attention scores per batch               |
| `attn_weights`  | (b, num_tokens, num_tokens) | Normalized attention weights per batch   |
| `values`        | (b, num_tokens, d_out)      | Projected values per batch               |
| `context_vecs`  | (b, num_tokens, d_out)      | Output context vectors per batch         |

In [ ]:
torch.manual_seed(123)

context_length = batch.shape[
    1
]  # Context length is the number of tokens in the input sequence.
d_in = batch.shape[-1]  # Input embedding size.
d_out = 2  # Output embedding size.

# Create an instance of the CausalAttention class.
ca = CausalAttention(d_in=d_in, d_out=d_out, context_length=context_length, dropout=0.0)

# Calcuate context vectors for the batch of inputs.
context_vecs = ca(batch)
print(f"Context vectors for the batch of inputs:\n{context_vecs}\n")
print(f"Shape of context vectors: {context_vecs.shape}")

__How does the class know that the inputs are in batches?__

The class knows the inputs are in batches because of the shape of the input tensor you pass to it.

When you call:

```python
context_vecs = ca(batch)
```

your `batch` tensor has shape (batch_size, num_tokens, d_in), for example `(2, 6, 3)`.

Inside the `forward` method:

```python
b, num_tokens, d_in = x.shape
```

PyTorch automatically handles the first dimension `(b)` as the batch size. All operations (like linear layers and matrix multiplications) are __vectorized__ to work on each batch independently.

- The `nn.Linear` layers (`self.W_query`, etc.) are designed to process the batch dimension automatically.
- Matrix multiplications like `queries @ keys.transpose(1, 2)` are performed for each batch in parallel.

_Summary:_

The class doesn't need any special code to "know" about batches—PyTorch's tensor operations and layers always treat the first dimension as the batch, and process all batches in parallel. The shape of the input tells the class how many batches there are.

Multi-head attention

<div>
<img src="_docs/multihead-attention-01.png" width="800"/><br><br>
<img src="_docs/multihead-attention-02.png" width="800"/>
</div>

In [ ]:
# Wrapper class to implement multi-head attention.
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            modules=[
                CausalAttention(
                    d_in=d_in,
                    d_out=d_out,
                    context_length=context_length,
                    dropout=dropout,
                    qkv_bias=qkv_bias,
                )
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        # Concatenate context vectors from all heads along the last dimension.
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
print(f"Input batch:\n{batch}\n")
print(f"Shape of the input batch: {batch.shape}")

In [ ]:
torch.manual_seed(14)
context_length = batch.shape[1]  # Number of tokens.
d_in, d_out = 3, 2
num_heads = 3  # Number of attention heads.

# Create an instance of the MultiHeadAttention class.
mhaw = MultiHeadAttentionWrapper(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0,
    num_heads=num_heads,
)

# Calculate context vectors for the batch of inputs.
context_vecs = mhaw(batch)
print(f"Context vectors for the batch of inputs:\n{context_vecs}\n")
print(f"Shape of context vectors: {context_vecs.shape}")

Multi-head attention with weight splits

<div>
<img src="_docs/multihead-attention-03.png" width="800"/>
</div>

In [ ]:
torch.manual_seed(14)

b = 2  # Batch size.
context_length = 4  # Number of tokens.
d_in = d_out = 6  # Input and output embedding sizes.
test_input = torch.randn(b, context_length, d_in).round(decimals=2)
print(f"Input tensor shape: {test_input.shape}")
print(f"Input tensor:\n{test_input}\n")

In [ ]:
torch.manual_seed(14)

num_heads = 2  # Number of attention heads.
dim_head = d_out // num_heads  # Output embedding size per head.

W_query = torch.randn(d_in, d_out).round(decimals=2)
W_key = torch.randn(d_in, d_out).round(decimals=2)
W_value = torch.randn(d_in, d_out).round(decimals=2)

print(f"Each head's dimension: {d_in, dim_head}\n")
print(f"Test query shape: {W_query.shape}")
print(f"Test query:\n{W_query}\n")
print(f"Test key shape: {W_key.shape}")
print(f"Test key:\n{W_key}\n")
print(f"Test value shape: {W_value.shape}\n")
print(f"Test value:\n{W_value}\n")

In [ ]:
test_query = test_input @ W_query
test_keys = test_input @ W_key
test_values = test_input @ W_value

print(f"Test query shape: {test_query.shape}")
print(f"Test query:\n{test_query}\n")
print(f"Test keys shape: {test_keys.shape}")
print(f"Test keys:\n{test_keys}\n")
print(f"Test values shape: {test_values.shape}\n")
print(f"Test values:\n{test_values}\n")


In [ ]:
# Reshape the test query, keys, and values for multi-head attention.
query = test_query.reshape(b, context_length, num_heads, dim_head)
keys = test_keys.reshape(b, context_length, num_heads, dim_head)
values = test_values.reshape(b, context_length, num_heads, dim_head)

print(f"Test query shape after reshaping: {query.shape}")
print(f"Test query after reshaping:\n{query}\n")
print(f"Test keys shape after reshaping: {keys.shape}")
print(f"Test keys after reshaping:\n{keys}\n")
print(f"Test values shape after reshaping: {values.shape}")
print(f"Test values after reshaping:\n{values}\n")

A __contiguous tensor__ in PyTorch means that the data is stored in a single, continuous chunk of memory, with no gaps or jumps between elements. This is important for some operations (like `.view()`) that rely on the underlying memory layout.

- _Why does it matter?_

    Some tensor operations (like slicing, transposing, or certain advanced indexing) can create non-contiguous tensors. These tensors still work for most operations, but some methods (like `.view()`) require the tensor to be contiguous for efficiency and correctness.

- _How to check?_

    You can check if a tensor is contiguous with `.is_contiguous()`:

    ```python
    print(tensor.is_contiguous())
    ```

- _How to make a tensor contiguous?_

    Use `.contiguous()` to get a contiguous copy:
    ```python
    tensor = tensor.contiguous()
    ```

- _Why does `.reshape()` work?_

    `.reshape()` is more flexible: if the tensor is not contiguous, it will automatically make a contiguous copy if needed, so it works in more cases than `.view()`.

__Summary:__

A contiguous tensor has its elements stored in an unbroken block of memory. Some operations require this for efficiency. If you get a "not contiguous" error, use `.reshape()` or `.contiguous()` to fix it.

In [ ]:
# Reshape the test query, keys, and values for multi-head attention.
query = test_query.contiguous().view(b, context_length, num_heads, dim_head)
keys = test_keys.contiguous().view(b, context_length, num_heads, dim_head)
values = test_values.contiguous().view(b, context_length, num_heads, dim_head)

print(f"Test query shape after reshaping: {query.shape}")
print(f"Test query after reshaping:\n{query}\n")
print(f"Test keys shape after reshaping: {keys.shape}")
print(f"Test keys after reshaping:\n{keys}\n")
print(f"Test values shape after reshaping: {values.shape}")
print(f"Test values after reshaping:\n{values}\n")

In [ ]:
# Transposes from shape (b, context_length, num_heads, dim_head) to (b, num_heads, context_length, dim_head)
query = test_query.view(b, context_length, num_heads, dim_head).transpose(1, 2)
keys = test_keys.view(b, context_length, num_heads, dim_head).transpose(1, 2)
values = test_values.view(b, context_length, num_heads, dim_head).transpose(1, 2)

print(f"Test query shape after reshaping: {query.shape}")
print(f"Test query after reshaping:\n{query}\n")
print(f"Test keys shape after reshaping: {keys.shape}")
print(f"Test keys after reshaping:\n{keys}\n")
print(f"Test values shape after reshaping: {values.shape}")
print(f"Test values after reshaping:\n{values}\n")

In [ ]:
# Attention scores calculation.
attn_scores = query @ keys.transpose(-2, -1)  # Transpose the last two dimensions.
print(f"Attention scores shape: {attn_scores.shape}")
print(f"Attention scores:\n{attn_scores}\n")

In [ ]:
# Masking and softmax normalisation are not applied in this example.
# For simplicity, assume 'attn_scores' = 'attn_weights'.
attn_weights = attn_scores

# Calculate context vectors.
context_vecs = attn_weights @ values
print(f"Context vectors shape (w/o transpose): {context_vecs.shape}")
print(f"Context vectors (w/o transpose):\n{context_vecs}\n")

In [ ]:
context_vecs = (attn_weights @ values).transpose(1, 2)
print(f"Context vectors shape (w/ transpose): {context_vecs.shape}")
print(f"Context vectors (w/ transpose):\n{context_vecs}\n")

In [ ]:
context_vecs = (attn_weights @ values).transpose(1, 2)

# Combine heads, d_out = num_heads * dim_head.
context_vecs = context_vecs.contiguous().view(
    b, context_length, d_out
)  # Flatten the last two dimensions.
print(f"Context vectors shape (after combining heads): {context_vecs.shape}")
print(f"Context vectors (after combining heads):\n{context_vecs}\n")

In [ ]:
print(f"'query' shape (b, num_heads, context_length, dim_head): {query.shape}\n")

print(f"'keys' shape (b, num_heads, context_length, dim_head): {keys.shape}")
print(
    f"'keys.transpose(-2, -1)' shape (b, num_heads, dim_head, context_length): {keys.transpose(-2, -1).shape}\n"
)

print(f"'values' shape (b, num_heads, context_length, dim_head): {values.shape}\n")

print(
    f"'attn_scores' shape (b, num_heads, context_length, context_length): {attn_scores.shape}\n"
)

print(
    f"'context_vecs' shape, before transpose, (b, num_heads, context_length, dim_head): {(attn_weights @ values).shape}"
)
print(
    f"'context_vecs' shape, after transpose, (b, context_length, num_heads, dim_head): {(attn_weights @ values).transpose(1, 2).shape}"
)
print(
    f"'context_vecs' shape, after combining heads, (b, context_length, d_out): {context_vecs.shape}"
)

In [ ]:
# An efficient implementation of multi-head attention.
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.context_length = context_length
        self.num_heads = num_heads
        self.dim_head = (
            d_out // num_heads
        )  # Reduces the projection dim to match the desired output dim.

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(
            d_out, d_out
        )  # Use a Linear layer to combine head outputs.
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # # Compute query, key, and value vectors.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Implicitly split the matrix by adding a num_heads dimension.
        # Then unroll the last dim: (b, num_tokens, d_out) --> (b, num_tokens, num_heads, head_dim).
        keys = keys.view(b, num_tokens, self.num_heads, self.dim_head)
        values = values.view(b, num_tokens, self.num_heads, self.dim_head)
        queries = queries.view(b, num_tokens, self.num_heads, self.dim_head)

        # Transpose from shape (b, num_tokens, num_heads, dim_head) to (b, num_heads, num_tokens, dim_head).
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute attention scores.
        attn_scores = queries @ keys.transpose(
            2, 3
        )  # Compute dot product for each head.
        mask_bool = self.mask.bool()[
            :num_tokens, :num_tokens
        ]  # Mask truncated to the number of tokens.

        attn_scores.masked_fill_(
            mask_bool, -torch.inf
        )  # Use the mask to fill attention scores.

        # Normalise attention scores.
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        # Apply dropout to attention weights.
        attn_weights = self.dropout(attn_weights)

        # Compute context vectors.
        context_vecs = (attn_weights @ values).transpose(
            1, 2
        )  # Tensor shape: (b, num_tokens, n_heads, dim_head).
        context_vecs = context_vecs.contiguous().view(
            b, num_tokens, self.d_out
        )  # Combine heads, where self.d_out = self.num_heads * self.dim_head
        context_vecs = self.out_proj(context_vecs)  # Add an optional linear projection.

        return context_vecs


In [ ]:
torch.manual_seed(14)

# Define inputs for the multi-head attention class.
batch_size, context_length, d_in = (
    batch.shape
)  # Batch size, context length, and input embedding size.
d_out = 6  # Output embedding size.
num_heads = 2  # Number of attention heads.
dropout = 0.0  # Dropout probability.

# Create an instance of the MultiHeadAttention class.
mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=dropout,
    num_heads=num_heads,
)

# Calculate context vectors for the batch of inputs.
context_vecs = mha(batch)

print(f"Shape of input batch: {batch.shape}")
print(f"Input batch:\n{batch}\n")

print(f"Shape of context vectors: {context_vecs.shape}")
print(f"Context vectors for the batch of inputs:\n{context_vecs}")

## 4 Implement a GPT model
- #### Step #3 - LLM architecture

### _4.1 Coding an LLM architecture_

<div>
<img src="_docs/41-llm-architecture-01.png" width="800"/><br><br>
<img src="_docs/41-llm-architecture-02.png" width="800"/><br><br>
<img src="_docs/41-llm-architecture-03.png" width="800"/><br><br>
<img src="_docs/41-llm-architecture-04.png" width="800"/>
</div>

Configuration of the small GPT-2 model.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of layers (transformer blocks)
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Query-Key-Value bias
}

A placeholder GPT model architecture class.

In [ ]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # Uses a placeholder for TransformerBlock.
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # Uses a placeholder for LayerNorm.
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        # Output head for logits.
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits


# A simple placeholder class that will be replaced by a real TransformerBlock later.
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    # Placeholder forward method that does nothing and returns the input as is.
    def forward(self, x):
        return x


# A simple placeholder class that will be replaced by a real LayerNorm later.
class DummyLayerNorm(nn.Module):
    # The parameters here just to mimick the real LayerNorm interface.
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(f"Input batch:\n{batch}")

In [ ]:
torch.manual_seed(123)

model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
print(f"Output logits:\n{logits}")

In [ ]:
# tokenizer.decode_batch(
#     logits.argmax(dim=-1).tolist()
# )  # Decode the predicted token indices back to text.
# print("Decoded output:", tokenizer.decode_batch(logits.argmax(dim=-1).tolist()))

### _4.2 Layer normalization_

Normalizing activations with layer normalization

<div>
<img src="_docs/42-layer-normalization-01.png" width="800"/><br><br>
<img src="_docs/42-layer-normalization-02.png" width="800"/><br><br>
<img src="_docs/42-layer-normalization-03.png" width="800"/>
</div>

In [ ]:
torch.manual_seed(123)

# Creates two training examples with five dimensions (features) each.
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(f"Input batch example:\n{batch_example}\n")
print(f"Output after passing through the layer:\n{out}\n")

# Calculate mean and variance of the input and output.
mean_in = batch_example.mean(dim=-1, keepdim=True)
var_in = batch_example.var(dim=-1, keepdim=True)

mean_out = out.mean(dim=-1, keepdim=True)
var_out = out.var(dim=-1, keepdim=True)

print(f"'batch_example':\n\tMean = {mean_in}\n\tVariance = {var_in}\n")
print(f"'out':\n\tMean = {mean_out}\n\tVariance = {var_out}\n")

# Normalize the layer outputs.
out_norm = (out - mean_out) / torch.sqrt(var_out)
mean_norm = out_norm.mean(dim=-1, keepdim=True)
var_norm = out_norm.var(dim=-1, keepdim=True)

print(f"Normalized layer outputs:\n\t{out_norm}\n")
print(f"'out_norm':\n\tMean = {mean_norm}\n\tVariance = {var_norm}\n")

torch.set_printoptions(sci_mode=False)
print(f"'out_norm':\n\tMean = {mean_norm}\n\tVariance = {var_norm}")


A layer normalization class.

In [ ]:
# Define layer normalization class.
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

mean_ln = out_ln.mean(dim=-1, keepdim=True)
var_ln = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print(f"Input batch example:\n{batch_example}\n")
print(f"'batch_example':\n\tMean = {mean_in}\n\tVariance = {var_in}\n")
print(f"'out_ln':\n\tMean = {mean_ln}\n\tVariance = {var_ln}")

```
Layer normalization vs. batch normalization.

If you are familiar with batch normalization, a common and traditional normalization method for neural networks, you may wonder how it compares to layer normalization. Unlike batch normalization, which normalizes across the batch dimension, layer normalization normalizes across the feature dimension. LLMs often require significant computational resources, and the available hardware or the specific use case can dictate the batch size during training or inference. Since layer normalization normalizes each input independently of the batch size, it offers more flexibility and stability in these scenarios. This is particularly beneficial for distributed training or when deploying models in environments where resources are constrained.
```

### _4.3 Feed forward network with GELU activations_

An implementation of the GELU activation function.

<div>
<img src="_docs/43-feed-forward-01.png" width="800"/><br><br>
<img src="_docs/43-feed-forward-02.png" width="800"/><br><br>
<img src="_docs/43-feed-forward-03.png" width="800"/><br><br>
<img src="_docs/43-feed-forward-04.png" width="800"/><br><br>
<img src="_docs/43-feed-forward-05.png" width="800"/>
</div>

In [ ]:
# Define a GELU activation function class.
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        _gelu = (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )
        return _gelu

In [ ]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

# Create 100 sample data points in the range –3 to 3.
x = torch.linspace(-3, 3, 1000)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print(f"{GPT_CONFIG_124M = }")

In [ ]:
# Define a feed forward neural network module.
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(
                cfg["emb_dim"], 4 * cfg["emb_dim"]
            ),  # Expand the embedding dimension.
            GELU(),  # Apply GELU activation function.
            nn.Linear(
                4 * cfg["emb_dim"], cfg["emb_dim"]
            ),  # Reduce the embedding dimension back to the original size.
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# Initialize the feed forward neural network with the GPT configuration.
ffn = FeedForward(cfg=GPT_CONFIG_124M)
# Create a sample input with batch dimension 2.
x = torch.rand(2, 3, 768)
out = ffn(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")

### _4.4 Shortcut connections_

<div>
<img src="_docs/44-shortcut-connection-01.png" width="800"/><br><br>
</div>


In [ ]:
# Createneural network to illustrate shortcut connections.
class ExampleDeepNN(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        # Implement 5 Layers.
        self.layers = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(in_features=layer_sizes[0], out_features=layer_sizes[1]),
                    nn.ReLU(),
                ),
                nn.Sequential(
                    nn.Linear(in_features=layer_sizes[1], out_features=layer_sizes[2]),
                    nn.ReLU(),
                ),
                nn.Sequential(
                    nn.Linear(in_features=layer_sizes[2], out_features=layer_sizes[3]),
                    nn.ReLU(),
                ),
                nn.Sequential(
                    nn.Linear(in_features=layer_sizes[3], out_features=layer_sizes[4]),
                    nn.ReLU(),
                ),
                nn.Sequential(
                    nn.Linear(in_features=layer_sizes[4], out_features=layer_sizes[5]),
                    nn.ReLU(),
                ),
            ]
        )

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer.
            layer_output = layer(x)

            # Check if shortcut can be applied.
            if self.use_shortcut and layer_output.shape == x.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

In [ ]:
# Implement a function to compute the gradients in the model’s backward pass.
def compute_gradients(model, input, target):
    # Forward pass.
    output = model(input)

    # Compute loss.
    loss = nn.MSELoss()(output, target)

    # Backward pass.
    loss.backward()

    for name, params in model.named_parameters():
        if "weight" in name:
            print(
                f"{name} - Gradient (mean absolute value):\n{params.grad.abs().mean().item()}"
            )

    # Return the gradients.
    return [
        param.grad for _, param in model.named_parameters() if param.grad is not None
    ]


In [ ]:
# Specify random seed for the initial weights for reproducibility.
torch.manual_seed(123)

# Define the layer sizes and a sample input tensor.
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1.0, 0.0, -1.0]])

# Create the model without shortcut connections.
model_without_shortcut = ExampleDeepNN(layer_sizes=layer_sizes, use_shortcut=False)

# Compute gradients without shortcut connections.
gradients_without_shortcut = compute_gradients(
    model=model_without_shortcut, input=sample_input, target=torch.tensor([[0.0]])
)
print(f"\nGradients without shortcut connections:\n{gradients_without_shortcut}")

Visualizing and understanding gradient output __without__ `shortcut connection`.
```
[
    # Layer #1 (size=3).
    tensor([
        [ 0.0009,  0.0000, -0.0009],
        [ 0.0000,  0.0000,  0.0000],
        [-0.0022,  0.0000,  0.0022]
    ]), # gradient - weight
    tensor([ 0.0009,  0.0000, -0.0022]), # gradient - bias
    
    # Layer #2 (size=3).
    tensor([
        [ 0.0000,  0.0000,  0.0000],
        [ 0.0050,  0.0000,  0.0026],
        [-0.0063,  0.0000, -0.0033]
    ]), # gradient - weight
    tensor([ 0.0000,  0.0161, -0.0202]), # gradient - bias
    
    # Layer #3 (size=3).
    tensor([
        [ 0.0000, -0.0007, -0.0021],
        [ 0.0000,  0.0070,  0.0222],
        [ 0.0000, -0.0006, -0.0018]
    ]), # gradient - weight
    tensor([-0.0037,  0.0382, -0.0032]), # gradient - bias
    
    # Layer #4 (size=3).
    tensor([[
        0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0085, 0.0178, 0.0084]
    ]), # gradient - weight
    tensor([0.0000, 0.0000, 0.0698]), 
    
    # Layer #5 (size=1).
    tensor([[0.0000, 0.0000, 0.0744]]), # gradient - weight
    tensor([0.3405])] # gradient - bias
```

In [ ]:
torch.manual_seed(123)

# Create the model with shortcut connections.
model_with_shortcut = ExampleDeepNN(layer_sizes=layer_sizes, use_shortcut=True)

# Compute gradients with shortcut connections.
gradients_with_shortcut = compute_gradients(
    model=model_with_shortcut, input=sample_input, target=torch.tensor([[0.0]])
)
print(f"\nGradients with shortcut connections:\n{gradients_with_shortcut}")

Visualizing and understanding gradient output __with__ `shortcut connection`.
```
[
    # Layer #1 (size=3).
    tensor([
        [ 1.9628,  0.0000, -1.9628],
        [ 0.0000,  0.0000,  0.0000],
        [-0.5381,  0.0000,  0.5381]
    ]), # gradient - weight
    tensor([ 1.9628,  0.0000, -0.5381]), # gradient - bias
    
    # Layer #2 (size=3).
    tensor([
        [ 0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000],
        [-0.5017,  0.0000,  0.3205]
    ]), # gradient - weight
    tensor([ 0.0000,  0.0000, -0.3821]), # gradient - bias
    
    # Layer #3 (size=3).
    tensor([
        [ 1.9119,  0.0000, -1.1690],
        [ 1.7656,  0.0000, -1.0795],
        [ 0.7425,  0.0000, -0.4540]
    ]), # gradient - weight
    tensor([1.4563, 1.3450, 0.5656]), # gradient - bias
    
    # Layer #4 (size=3).
    tensor([
        [ 0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000],
        [ 0.8384,  0.8586, -0.2571]
    ]), # gradient - weight
    tensor([0.0000, 0.0000, 0.5926]), # gradient - bias
    
    # Layer #5 (size=3).
    tensor([
        [4.0909, 4.1894, 1.1419]
    ]), # gradient - weight
    tensor([2.8915])] # gradient - bias
```

### _4.5 Transformer block_

<div>
<img src="_docs/45-transformer-block-01.png" width="800"/><br><br>
<img src="_docs/45-transformer-block-02.png" width="800"/>
</div>

`MultiHeadAttention`, `LayerNorm` and `FeedForward` classes have been defined previously.


In [ ]:
# Print the GPT configuration.
print(f"{GPT_CONFIG_124M = }")

In [ ]:
# Transformer block class.
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg=cfg)
        self.norm1 = LayerNorm(emb_dim=cfg["emb_dim"])
        self.norm2 = LayerNorm(emb_dim=cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(p=cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block.
        shortcut = x

        x = self.norm1(x)
        x = self.att(x)  # shape: (batch_size, num_tokens, emb_dim)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back.

        # Shortcut connection for feed forward block.
        shortcut = x

        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back.

        return x

In [ ]:
torch.manual_seed(123)

# Create sample input of shape [batch_size, num_tokens, emb_dim].
x = torch.rand(2, 4, 768)

block = TransformerBlock(cfg=GPT_CONFIG_124M)
output = block(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

### _4.6 GPT2 model_

<div>
<img src="_docs/46-gpt2-model-01.png" width="800"/><br><br>
<img src="_docs/46-gpt2-model-02.png" width="1000"/><br><br>
<img src="_docs/46-gpt2-model-03.png" width="1000"/><br><br>
<img src="_docs/46-gpt2-model-04.png" width="1000"/>
</div>

In [ ]:
# GPT model class.
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(
            num_embeddings=cfg["vocab_size"], embedding_dim=cfg["emb_dim"]
        )
        self.pos_emb = nn.Embedding(
            num_embeddings=cfg["context_length"], embedding_dim=cfg["emb_dim"]
        )
        self.drop_emb = nn.Dropout(p=cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(emb_dim=cfg["emb_dim"])
        self.out_head = nn.Linear(
            in_features=cfg["emb_dim"], out_features=cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )  # Device setting for model training, CPU or GPU.
        x = tok_embeds + pos_embeds  # shape: (batch_size, seq_len/num_tokens, emb_dim)
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

In [ ]:
# Print the GPT configuration.
# print(f"{GPT_CONFIG_124M = }")
GPT_CONFIG_124M

In [ ]:
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
out = model(batch)

print(f"Input shape: {batch.shape}")
print(f"Input batch:\n{batch}\n")
print(f"Output shape: {out.shape}")
print(f"Output logits:\n{out}\n")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}\n")

print(f"Token embedding layer shape: {model.tok_emb.weight.shape}")
print(f"Output layer shape: {model.out_head.weight.shape}\n")

total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(
    f"Number of trainable parameters considering `weight tying`: {total_params_gpt2:,}\n"
)

total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"Total size of the model: {total_size_mb:.2f} MB")

In [ ]:
# List total model parameters and their shapes.
model_parameters = list(model.parameters())
print(f"Total number of parameters in the model: {len(model_parameters)}\n")
print("Shapes of the model parameters:")
for i, param in enumerate(model_parameters):
    print(f"\tParam {i + 1}: {param.shape}")

### _4.7 Generate Text_

<div>
<img src="_docs/47-generate-text-01.png" width="800"/><br><br>
<img src="_docs/47-generate-text-02.png" width="800"/><br><br>
<img src="_docs/47-generate-text-03.png" width="800"/><br><br>
</div>

In [ ]:
# Function to generate text using the GPT model.
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is a (batch, n_tokens) array of indices in the current context.
    for _ in range(max_new_tokens):
        # Crop current context, if it exceeds the supported context size.
        # e.g., if LLM supports only 5 tokens and the context size is 10, only the last 5 tokens are used as context.
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)  # shape (batch, n_tokens, vocab_size)

        # Focus only on the last time step.
        logits = logits[:, -1, :]  # shape (batch, vocab_size)

        # Apply softmax to convert logits to probabilities.
        probas = torch.softmax(logits, dim=-1)  # shape (batch, vocab_size)

        # Greedy decoding, get the index of the token with the highest probability.
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # shape (batch, 1)

        # Append sampled index to the running sequence.
        idx = torch.cat((idx, idx_next), dim=1)  # shape (batch, n_tokens+1)
    return idx

In [ ]:
# Generate text using the GPT model.
start_context = "Hello, I am"
encoded = tokenizer.encode(
    start_context
)  # Encode the starting context to token indices.
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

print(f"Input text: '{start_context}'")
print(f"Input text encoded: {encoded}")
print(f"'encoded_tensor.shape': {encoded_tensor.shape}\n")

# Generate 6 new tokens based on the starting context.
model.eval()
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"],
)
# print(GPT_CONFIG_124M["context_length"])

# Decode the generated token indices back to text.
decoded_text = tokenizer.decode(out.squeeze(0).tolist())

print(f"Generated tokens: {out}")
print(f"Length of generated tokens: {len(out[0])}\n")
print(f"Generated text output - decoded: '{decoded_text}'")

In [ ]:
# TODO - Images from the below section are added correctly.

# STAGE 2 - Foundation model

## Step #4 - Pretraining

### 5 Pretraining an LLM

<div>
<img src="_docs/50-pretraining-llm-01.png" width="800"/>
</div>

#### _5.1 Evaluating generative text models_

<div>
<img src="_docs/51-evaluation-generative-model-01.png" width="800"/><br><br>
<img src="_docs/51-evaluation-generative-model-02.png" width="800"/>
</div>

`GPTModel` and `generate_text_simple` are defined in section `4.6` and `4.7` respectively.

In [ ]:
import torch

torch.manual_seed(123)

# Define a smaller GPT configuration for faster generation.
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,  # shorten the context length from 1,024 to 256 tokens.
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

# Initialize the GPT model with the smaller configuration.
model = GPTModel(GPT_CONFIG_124M)
model.eval()

In [ ]:
import tiktoken

# https://github.com/openai/tiktoken/blob/main/tiktoken/core.py

# Example usage of the helper functions.
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")


# Helper functions to convert text to token IDs and vice versa.
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text=text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(
        0
    )  # .unsqueeze(0) adds the batch dimension
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # .squeeze(0) removes the batch dimension
    return tokenizer.decode(tokens=flat.tolist())


# Convert text to token IDs.
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)
print(f"Input text: '{start_context}'")
print(f"Input text encoded: {tokenizer.encode(start_context)}\n")
print(f"Output text: {token_ids_to_text(token_ids, tokenizer)}")

<div>
<img src="_docs/51-evaluation-generative-model-03.png" width="800"/>
</div>

In [ ]:
# Define input and target tokens for training.
inputs = torch.tensor(
    [
        [16833, 3626, 6100],  # ["every effort moves",
        [40, 1107, 588],  # "I really like"]
    ]
)
targets = torch.tensor(
    [
        [3626, 6100, 345],  # [" effort moves you",
        [1107, 588, 11311],  # " really like chocolate"]
    ]
)

# Disable gradtient tracking, for now.
with torch.no_grad():
    logits = model(inputs)

# Calculate probability of each token in vocabulary, using softmax.
probas = torch.softmax(logits, dim=-1)

print(f"Input tokens:\n{inputs}\n")
print(f"Target tokens:\n{targets}\n")
print(f"Logits shape: {logits.shape}")
print(f"Logits:\n{logits}\n")
print(f"Probabilities shape: {probas.shape}")
print(f"Probabilities:\n{probas}\n")

print("'logits.shape' = (batch_size, num_tokens, vocab_size)")

In [ ]:
# Get the token IDs with the highest probabilities.
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print(f"Token IDs:\n{token_ids}\n")

print(
    f"Targets batch 1: {token_ids_to_text(token_ids=targets[0], tokenizer=tokenizer)}"
)
print(
    f"Outputs batch 1: {token_ids_to_text(token_ids=token_ids[0].flatten(), tokenizer=tokenizer)}"
)

<div>
<img src="_docs/51-evaluation-generative-model-04.png" width="800"/><br><br>
<img src="_docs/51-evaluation-generative-model-05.png" width="800"/>
</div>

In [ ]:
# Print the probabilities of the target tokens for each text in the batch.
print("Probabilities of the target tokens for each text in the batch:")
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print(f"\tText 1: {target_probas_1}")

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print(f"\tText 2: {target_probas_2}\n")

print(f"{probas.shape = }")

`probas[text_idx, [0, 1, 2], targets[0]]`

`probas[text_idx, [0, 1, 2], [3626, 6100,  345]]`

Here’s what the code block is doing:

- It selects a specific text sample (by index) from a batch of predictions (`probas`) and targets.

- For text 1 (`text_idx = 0`), it extracts the predicted probabilities for the first three positions (`[0, 1, 2]`) and, at each position, selects the probability corresponding to the true target token (`targets[text_idx]`) at that position.

- It prints these probabilities for text 1.

- It repeats the same process for text 2 (`text_idx = 1`).

In summary:

- This code prints, for two different texts, the model’s predicted probability for the correct (`target`) token at the first three positions of each text. This is useful for evaluating how confident the model is in its predictions for the correct answer at each step.

<div>
<img src="_docs/51-evaluation-generative-model-06.png" width="800"/>
</div>

The goal is to get the average log probability as close to 0 as possible by updating the model’s weights as part of the training process. However, in deep learning, the common practice isn’t to push the average log probability up to 0 but rather to bring the negative average log probability down to 0. The negative average log probability is simply the average log probability multiplied by –1.

In deep learning, the term for turning this negative value, –10.7940, into 10.7940, is known as the cross entropy loss. PyTorch comes in handy here, as it already has a built-in cross_entropy function that takes care of all these six steps.

In [ ]:
# Compute the log probabilities of the target tokens for both texts in the batch.
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(f"{log_probas = }")

# Calculate the average log probability across all target tokens in the batch.
avg_log_probas = torch.mean(log_probas)
print(f"{avg_log_probas = }")

# Calculate the negative average log probability, negative log likelihood.
neg_avg_log_probas = avg_log_probas * -1
print(f"{neg_avg_log_probas = }")

_Perplexity_

Perplexity is a measure often used alongside cross entropy loss to evaluate the performance of models in tasks like language modeling. It can provide a more interpretable way to understand the uncertainty of a model in predicting the next token in a sequence.

Perplexity measures how well the probability distribution predicted by the model matches the actual distribution of the words in the dataset. Similar to the loss, a lower perplexity indicates that the model predictions are closer to the actual distribution.

Perplexity can be calculated as `perplexity = torch.exp(loss)`, which returns `tensor(48725.8203)` when applied to the previously calculated loss.

Perplexity is often considered more interpretable than the raw loss value because it signifies the effective vocabulary size about which the model is uncertain at each step. In
the given example, this would translate to the model being unsure about which among 48,725 tokens in the vocabulary to generate as the next token.

`Lower perlexity score = Better predictions`

In [ ]:
# Calculate the cross-entropy loss using PyTorch's built-in function.
print(f"Logits shape: {logits.shape}")
print(f"Targets shape: {targets.shape}\n")

# Flatten the logits and targets tensors for loss computation.
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print(f"Flattened logits: {logits_flat.shape}")
print(f"Flattened targets: {targets_flat.shape}\n")

# Compute the cross-entropy loss.
loss = torch.nn.functional.cross_entropy(input=logits_flat, target=targets_flat)
print(f"Cross-entropy loss: {loss.item()}\n")

# Compute perplexity.
perplexity = torch.exp(loss)
print(f"Perplexity: {perplexity.item()}")

Calculate the training and validation set losses.

<div>
<img src="_docs/51-evaluation-generative-model-07.png" width="800"/><br><br>
<img src="_docs/51-evaluation-generative-model-08.png" width="800"/>
</div>

In [ ]:
# Load text data from a file.
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

# Initialize the tokenizer.
tokenizer = tiktoken.get_encoding(encoding_name="gpt2")

# Tokenize the text data
tokens = tokenizer.encode(text_data)

print(f"Total number of characters:     {len(text_data)}")
print(f"Total number of tokens:         {len(tokens)}\n")

print(f"First 99 characters: {text_data[:99]}\n")
print(f"First 10 tokens: {tokens[:10]}")

In [ ]:
# Split the data into training and validation sets.
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)

# from chapter02 import create_dataloader_v1

# Create data loaders for training and validation datasets.
dl_train = create_dataloader_v1(
    text=train_data,
    batch_size=2,
    context_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)
dl_val = create_dataloader_v1(
    text=val_data,
    batch_size=2,
    context_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0,
)

print("Train loader:")
for x, y in dl_train:
    print(x.shape, y.shape)
print("\nValidation loader:")
for x, y in dl_val:
    print(x.shape, y.shape)

In [ ]:
# Function to calculate loss for a batch of input and target tokens.
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        input=logits.flatten(0, 1), target=target_batch.flatten()
    )
    return loss


# Function to calculate average loss over a data loader.
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        # Iterate over all batches, if no fixed 'num_batches' is specified.
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader, if 'num_batches' exceeds the number of batches in the data loader.
        # if num_batches > len(data_loader):
        #     num_batches = len(data_loader)
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()  # Sum loss for each batch.
        else:
            break
    return total_loss / num_batches  # Average the loss over all batches.

In [ ]:
# Calculate training and validation loss.
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
# device = 'cpu' # Force using CPU for this example.
print(f"Using device: '{device}'\n")
model.to(device)

# Disable gradient tracking for loss calculation.
with torch.no_grad():
    train_loss = calc_loss_loader(data_loader=dl_train, model=model, device=device)
    val_loss = calc_loss_loader(data_loader=dl_val, model=model, device=device)

print(f"Training loss:      {train_loss}")
print(f"Validation loss:    {val_loss}")

<div>
<img src="_docs/51-evaluation-generative-model-09.png" width="800"/>
</div>

## Step #5 - Training loop

#### _5.2 Training an LLM_

<div>
<img src="_docs/52-training-llm-01.png" width="800"/>
</div>

In [ ]:
# Main function for pretraining an LLM.
def train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs,
    eval_freq,
    eval_iter,
    start_context,
    tokenizer,
):
    # Initialize lists to track losses and tokens seen.
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Start the main training loop.
    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode.

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # Reset loss gradients from the previous batch iteration.
            loss = calc_loss_batch(input_batch, target_batch, model, device)

            loss.backward()  # Calculate loss gradients.
            optimizer.step()  # Update model weights using loss gradients.
            tokens_seen += (
                input_batch.numel()
            )  # Count the number of tokens processed in this batch.
            global_step += 1

            # Optional evaluation step.
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(
                    f"Epoch {epoch + 1} (Step {global_step:06d}): Train loss {train_loss:.3f}, Val loss {val_loss:.3f}"
                )

        # Prints a sample text, after each epoch.
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen


# Helper function to evaluate the model on training and validation datasets.
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()  # Set the model to evaluation mode.

    # Disable gradient tracking for loss calculation.
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()  # Set the model back to training mode.
    return train_loss, val_loss


# Helper function to generate and print a sample text from the model.
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()

    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)

    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format.
    print("\n")
    model.train()

In [ ]:
%%time
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.0004, weight_decay=0.1)
num_epochs = 10

train_losses, val_losses, tokens_seen = train_model_simple(
    model,
    dl_train,
    dl_val,
    optimizer,
    device,
    num_epochs=num_epochs,
    eval_freq=10,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer,
)

<div>
<img src="_docs/52-training-llm-02.png" width="800"/>
</div>

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(8, 6))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()


epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

<div>
<img src="_docs/52-training-llm-03.png" width="800"/>
</div>

## Step #6 - Model evaluation

#### _5.3 Decoding strategies to control randomnes_

In [ ]:
model.to("cpu")
model.eval()

In [ ]:
# Generate text using the trained GPT model.
print("Generating text using the trained GPT model...\n")
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
)
print(f"Output text:\n{token_ids_to_text(token_ids, tokenizer)}")

Temperature scaling

<div>
<img src="_docs/53-decoding-strategies-01.png" width="800"/>
</div>

In [ ]:
# Example of calculating next token probabilities from logits.
vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
}
inverse_vocab = {v: k for k, v in vocab.items()}

next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()

print(f"Next token probabilities:\n{probas}\n")
print(f"Next token ID: {next_token_id}")
print(f"Next token: {inverse_vocab[next_token_id]}")

In [ ]:
# Example of sampling the next token from the probability distribution.
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(f"Next token: {inverse_vocab[next_token_id]}")

In [ ]:
# Sample multiple tokens to see the frequency distribution.
def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))

    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")


print_sampled_tokens(probas)

In [ ]:
# Visualize the effect of temperature scaling on the probability distribution.
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


# Original, lower, and higher confidence.
temperatures = [1, 0.1, 5]
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

x = torch.arange(len(vocab))
bar_width = 0.15
fig, ax = plt.subplots(figsize=(8, 6))

for i, T in enumerate(temperatures):
    rects = ax.bar(
        x + i * bar_width, scaled_probas[i], bar_width, label=f"Temperature = {T}"
    )
ax.set_ylabel("Probability")
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
next_token_logits2 = next_token_logits / 0.1
next_token_logits3 = next_token_logits / 5

probas2 = torch.softmax(next_token_logits2, dim=0)
probas3 = torch.softmax(next_token_logits3, dim=0)

print(f"Next token probabilities (T=0.1):\n{probas2}\n")
print(f"Next token probabilities (T=5):\n{probas3}")

Top-k sampling

<div>
<img src="_docs/53-decoding-strategies-02.png" width="800"/>
</div>

In [ ]:
# Example of top-k sampling from the logits.
top_k = 3
top_logits, top_pos = torch.topk(input=next_token_logits, k=top_k)
print(f"Next token logits: {next_token_logits}")
print(f"Top logits: {top_logits}")
print(f"Top positions: {top_pos}")

In [ ]:
new_logits = torch.where(
    # Identify logits less than the minimum in the top 3.
    condition=next_token_logits < top_logits[-1],
    # Assign '-inf' to these lower logits.
    input=torch.tensor(float("-inf")),
    # Retain the original logits for all other tokens.
    other=next_token_logits,
)
print(f"New logits (top-k filtered): {new_logits}")

# Calculate probabilities from the filtered logits.
topk_probas = torch.softmax(new_logits, dim=0)
print(f"Top-k probabilities: {topk_probas}")

<div>
<img src="_docs/53-decoding-strategies-03.png" width="800"/>
</div>

In [ ]:
# Function (modified) for text generation with more diversity, temperature scaling and top-k sampling.
def generate(
    model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None
):
    # The for loop is the same as before: gets logits and only focuses on the last time step.
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # Filter logits with top_k sampling.
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits
            )

        # Apply temperature scaling and sample the next token.
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # Carry out greedy next token selection as before, when temperature scaling is disabled.
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # Stop generating early, if end-of-sequence token is encountered.
        if idx_next == eos_id:
            break

        # Append sampled index to the running sequence.
        idx = torch.cat((idx, idx_next), dim=1)  # shape (batch, n_tokens+1)
    return idx

In [ ]:
# Set random seed for reproducibility.
torch.manual_seed(123)

# Generate text with temperature scaling and top-k sampling.
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4,
)
print(f"Generated text:\n{token_ids_to_text(token_ids, tokenizer)}")

## Step #7 - Load pretrained weights

#### _5.4 Loading and saving model weights in PyTorch_

<div>
<img src="_docs/54-loading-saving-model-01.png" width="800"/>
</div>

Saving an `LLM` model:
``` python
torch.save(model.state_dict(), "model.pth")
```

Loading an `LLM` model:
``` python
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()
```

In [ ]:
# Same as defined earlier.
optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.0004, weight_decay=0.1)

# Save model and optimizer state dictionaries.
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth",
)

# Load model state dictionaries.
checkpoint = torch.load("model_and_optimizer.pth", map_location=device)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

# Load optimizer state dictionaries.
optimizer = torch.optim.AdamW(params=model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

#### _5.5 Loading pretrained weights from OpenAI_

<div>
<img src="_docs/55-loading-openai-model-01.png" width="800"/>
</div>

In [ ]:
# Import TensorFlow and TQDM, since GPT-2 model used them.
import tensorflow as tf
import tqdm

print(f"TensorFlow version:     {tf.__version__}")
print(f"TQDM version:           {tqdm.__version__}")

In [ ]:
# Download the GPT-2 model code from GitHub.
import urllib.request

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch05/01_main-chapter-code/gpt_download.py"
filename = url.split("/")[-1]
urllib.request.urlretrieve(url, filename)

In [ ]:
# Download and load the GPT-2 architecture settings and weight parameters.
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

print(
    f"\nSettings: {settings}\n\n"
    f"Parameter dictionary keys: {params.keys()}\n\n"
    f"{params['wte'].shape = } --> token embedding weight tensor shape\n\n"
    f"{params['wpe'].shape = } --> position embedding weight tensor shape\n\n"
    f"{params['wte'] = } --> token embedding weight tensor\n"
)

In [ ]:
# Define model configurations for different GPT-2 sizes.
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Configure the model parameters based on the selected GPT-2 size.
model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

# Update context length and qkv_bias settings.
NEW_CONFIG.update({"context_length": 1024})
NEW_CONFIG.update({"qkv_bias": True})

# Initialize the GPT model with the new configuration.
gpt = GPTModel(NEW_CONFIG)
gpt.eval()

Parameter dictionary keys:
1. `wte` - token embeddings `(50257, 768)`

2. `wpe` - positional embeddings `(1024, 768)`

3. `Transformer-blocks` - 12 blocks
    - `layer normalisation #1`
        - `transformer/b0/ln_1/g` - scale
        - `transformer/b0/ln_1/b` - shift
    - `attention layer` - (Q, K, V)
        - `transformer/b0/attn/c_attn/w` - weight
        - `transformer/b0/attn/c_attn/b` - bias
    - `layer normalisation #2`
        - `transformer/b0/ln_2/g` - scale
        - `transformer/b0/ln_2/b` - shift
    - `feed-forward NN`
        - `transformer/b0/mlp/c_fc/w` - weight, fully connected layer
        - `transformer/b0/mlp/c_fc/b` - bias, fully connected layer

        - `transformer/b0/mlp/c_proj/w` - weight, projection layer
        - `transformer/b0/mlp/c_proj/b` - bias, projection layer
    - `output projection layer`
        - `transformer/b0/attn/c_proj/w` - weight

4. Final `layer normalisation`
    - `g` - scale
    - `b` - shift


In [ ]:
# Function to assign trainable PyTorch parameters (weights) from a tensor/array.
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

# Function to load GPT-2 weights into the GPT model.
# Set the model’s positional and token embedding weights to those specified in params.
def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    # Iterate over each transformer block and set the weights and biases.
    for i in range(len(params["blocks"])):
        # Divide the attention weights into three equal parts (query, key, and value).
        q_w, k_w, v_w = np.split(
            ary=params["blocks"][i]["attn"]["c_attn"]["w"],
            indices_or_sections=3,
            axis=-1,
        )
        gpt.trf_blocks[i].att.W_query.weight = assign(
            left=gpt.trf_blocks[i].att.W_query.weight, right=q_w.T
        )
        gpt.trf_blocks[i].att.W_key.weight = assign(
            left=gpt.trf_blocks[i].att.W_key.weight, right=k_w.T
        )
        gpt.trf_blocks[i].att.W_value.weight = assign(
            left=gpt.trf_blocks[i].att.W_value.weight, right=v_w.T
        )

        # Divide the bias weights into three equal parts (query, key, and value).
        q_b, k_b, v_b = np.split(
            ary=params["blocks"][i]["attn"]["c_attn"]["b"],
            indices_or_sections=3,
            axis=-1,
        )
        gpt.trf_blocks[i].att.W_query.bias = assign(
            left=gpt.trf_blocks[i].att.W_query.bias, right=q_b
        )
        gpt.trf_blocks[i].att.W_key.bias = assign(
            left=gpt.trf_blocks[i].att.W_key.bias, right=k_b
        )
        gpt.trf_blocks[i].att.W_value.bias = assign(
            left=gpt.trf_blocks[i].att.W_value.bias, right=v_b
        )

        # Set the output projection weights and biases.
        gpt.trf_blocks[i].att.out_proj.weight = assign(
            left=gpt.trf_blocks[i].att.out_proj.weight,
            right=params["blocks"][i]["attn"]["c_proj"]["w"].T,
        )
        gpt.trf_blocks[i].att.out_proj.bias = assign(
            left=gpt.trf_blocks[i].att.out_proj.bias,
            right=params["blocks"][i]["attn"]["c_proj"]["b"],
        )

        # Set the feed-forward network weights and biases.
        gpt.trf_blocks[i].ff.layers[0].weight = assign(
            left=gpt.trf_blocks[i].ff.layers[0].weight,
            right=params["blocks"][i]["mlp"]["c_fc"]["w"].T,
        )  # fully-connected layer (layer 1)
        gpt.trf_blocks[i].ff.layers[0].bias = assign(
            left=gpt.trf_blocks[i].ff.layers[0].bias,
            right=params["blocks"][i]["mlp"]["c_fc"]["b"],
        )  # fully-connected layer (layer 1)
        gpt.trf_blocks[i].ff.layers[2].weight = assign(
            left=gpt.trf_blocks[i].ff.layers[2].weight,
            right=params["blocks"][i]["mlp"]["c_proj"]["w"].T,
        )  # projection layer (layer 2)
        gpt.trf_blocks[i].ff.layers[2].bias = assign(
            left=gpt.trf_blocks[i].ff.layers[2].bias,
            right=params["blocks"][i]["mlp"]["c_proj"]["b"],
        )  # projection layer (layer 2)

        # Set the layer normalization scale and shift parameters.
        gpt.trf_blocks[i].norm1.scale = assign(
            left=gpt.trf_blocks[i].norm1.scale, right=params["blocks"][i]["ln_1"]["g"]
        )  # layer norm 1
        gpt.trf_blocks[i].norm1.shift = assign(
            left=gpt.trf_blocks[i].norm1.shift, right=params["blocks"][i]["ln_1"]["b"]
        )  # layer norm 1
        gpt.trf_blocks[i].norm2.scale = assign(
            left=gpt.trf_blocks[i].norm2.scale, right=params["blocks"][i]["ln_2"]["g"]
        )  # layer norm 2
        gpt.trf_blocks[i].norm2.shift = assign(
            left=gpt.trf_blocks[i].norm2.shift, right=params["blocks"][i]["ln_2"]["b"]
        )  # layer norm 2

    # Set the final layer normalization scale and shift parameters.
    gpt.final_norm.scale = assign(left=gpt.final_norm.scale, right=params["g"])
    gpt.final_norm.shift = assign(left=gpt.final_norm.shift, right=params["b"])

    # Final, linear output, layer weights.
    # The original GPT-2 model by OpenAI used 'weight tying' (reusing the token embedding weights in the output layer to reduce the total number of parameters).
    gpt.out_head.weight = assign(left=gpt.out_head.weight, right=params["wte"])

In [ ]:
# Load the GPT-2 weights into the GPT model.
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
torch.manual_seed(123)

# Generate text using the loaded GPT-2 model.
token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5,
)
print(f"Output text:\n{token_ids_to_text(token_ids, tokenizer)}")

# STAGE 3 - Fine-tuning (classifier/personal-assistant)

## Step #8 - Fine-tuning (classification)

### 6 Fine-tuning for classification

<div>
<img src="_docs/60-fine-tuning-classification.png" width="800"/>
</div>

#### _6.1 Different categories of fine tuning_

<div>

_6.1.1 Instruction fine-tuning_

<img src="_docs/61-fine-tuning-categories-01.png" width="800"/><br><br>

_6.1.2. Classification fine-tuning_

<img src="_docs/61-fine-tuning-categories-02.png" width="800"/>
</div>

#### _6.2 Preparing the dataset_

<div>
<img src="_docs/62-preparing-dataset-01.png" width="800"/><br><br>
<img src="_docs/62-preparing-dataset-02.png" width="800"/>
</div>

In [ ]:
# Download and unzip the dataset.
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Download the file.
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzip the file.
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add a '.tsv' file extension.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")


download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

In [ ]:
import pandas as pd

# Read the data as a DataFrame.
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
print(f"{df}\n")

print(df["Label"].value_counts())

In [ ]:
# Create a balanced dataset.
def create_balanced_dataset(df):
    # Count the instances of “spam”.
    num_spam = df[df["Label"] == "spam"].shape[0]

    # Randomly sample 'ham' instances to match the number of 'spam' instances.
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    # Combine 'ham' subset with 'spam'
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

# Convert the 'string' class labels 'ham' and 'spam' into integer class labels 0 and 1, respectively.
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
# Split the dataset.
def random_split(df, train_frac, validation_frac):

    # Shuffle the entire DataFrame.
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices.
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame.
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df


# Test size is implied to be 0.2 as the remainder.
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

print("Dataset sizes:")
print(f"Train      --> {len(train_df)}")
print(f"Validation --> {len(validation_df)}")
print(f"Test       --> {len(test_df)}")

#  Save the dataset as CSV files.
data_file_path = "sms_spam_collection"
os.makedirs(data_file_path, exist_ok=True)

train_df.to_csv(f"{data_file_path}/train.csv", index=None)
validation_df.to_csv(f"{data_file_path}/validation.csv", index=None)
test_df.to_csv(f"{data_file_path}/test.csv", index=None)

#### _6.3 Creating data-loaders_

<div>
<img src="_docs/63-creating-dataloaders-01.png" width="800"/><br><br>
</div>

In [ ]:
# Verify the token ID for the end-of-text token, which is used to indicate the end of a text sequence in GPT-2.
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
eot = "<|endoftext|>"
print(f" Token id for '{eot}': {tokenizer.encode(eot, allowed_special={eot})}")

In [ ]:
import torch
from torch.utils.data import Dataset

# Setting up a Pytorch Dataset class.
class SpamDataset(Dataset):
    """A PyTorch Dataset class for loading and encoding spam data."""

    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 1. Pretokenize texts
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

            # 2. Truncate sequences, if they are longer than max_length
            self.encoded_texts = [
                encoded_text[: self.max_length] for encoded_text in self.encoded_texts
            ]

        # 3. Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long),
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length


In [ ]:
path_data = "sms_spam_collection"
print(f"Data files in '{path_data}': {os.listdir(path_data)}\n")

# Pytorch datasets.
train_dataset = SpamDataset(
    csv_file=f"{path_data}/train.csv", max_length=None, tokenizer=tokenizer
)  # training
val_dataset = SpamDataset(
    csv_file=f"{path_data}/validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)  # validation
test_dataset = SpamDataset(
    csv_file=f"{path_data}/test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)  # test

print(f"Max sequence length in train dataset: {train_dataset.max_length}")

<div>
<img src="_docs/63-creating-dataloaders-02.png" width="800"/>
</div>

In [ ]:
# from torch.utils.data import DataLoader

num_workers = 0  # Ensures compatibility with most computers.
batch_size = 8
torch.manual_seed(123)

# PyTorch data-loaders.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)  # training
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)  # validation
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)  # test
print(
    f"Number of batches-\n"
    f"    train-loader:      {len(train_loader)}\n"
    f"    validation-loader: {len(val_loader)}\n"
    f"    test-loader:       {len(test_loader)}\n"
)

# Inspect the dimensions of the input and target batches from the train data-loader.
for input_batch, target_batch in train_loader:
    pass
print(
    f"Batch dimensions-\n"
    f"    Input batch: {input_batch.shape}\n"
    f"    Label batch: {target_batch.shape}"
)

#### _6.4 Initialising a model with pretrained weights_

<div>
<img src="_docs/64-initialise-model-pretrained-weights-01.png" width="800"/>
</div>

In [ ]:
# Model configuration and input prompt for text generation.
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size": 50257,        # Vocabulary size
    "context_length": 1024,     # Context length
    "drop_rate": 0.0,           # Dropout rate
    "qkv_bias": True,           # Query-key-value bias
}

# Model configurations for different GPT-2 sizes.
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Update the base configuration with the chosen model's specific settings.
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

print(f"Model configuration for '{CHOOSE_MODEL}':")
for key, value in BASE_CONFIG.items():
    print(f"    {key}: {value}")

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"\nDataset length '{train_dataset.max_length}' exceeds the model's context length '{BASE_CONFIG['context_length']}'.\n"
    f"Reinitialize dataset with 'max_length' <= {BASE_CONFIG['context_length']}."
)


In [ ]:
from gpt_download import download_and_load_gpt2
from utils import GPTModel, load_weights_into_gpt

# Extract the model size from the chosen model name and load the corresponding GPT-2 settings and parameters.
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size, models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
# Generate text using the loaded GPT-2 model.
text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

In [ ]:
# Validate whether the model already classifies spam messages by prompting it with instructions.
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

#### _6.5 Adding a classification head_

<div>
<img src="_docs/65-add-classification-head-01.png" width="800"/>
</div>

In [ ]:
# Print the model architecture to verify that the weights have been loaded correctly.
print(model)

In [ ]:
torch.manual_seed(123)

# Freeze the model parameters to prevent them from being updated during training, since we will only be training the new classification head.
for param in model.parameters():
    param.requires_grad = False

# Add a classification layer.
num_classes = 2
model.out_head = torch.nn.Linear(
    in_features=BASE_CONFIG["emb_dim"], 
    out_features=num_classes
)

<div>
<img src="_docs/65-add-classification-head-02.png" width="800"/><br><br>
</div>

In [ ]:
# Unfreeze the last transformer block and the final layer normalization, so that they can be fine-tuned for the spam classification task.
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

In [ ]:
# Use an example to test the model output.
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(f"Inputs               : {inputs}")
print(f"Inputs dimensions    : {inputs.shape}\n")    # shape: (batch_size, num_tokens)

# Pass the encoded token IDs to the model as usual.
with torch.no_grad():
    outputs = model(inputs)
print(f"Outputs               : {outputs}")
print(f"Outputs dimensions    : {outputs.shape}")

<div>
<img src="_docs/65-add-classification-head-03.png" width="800"/><br><br>
<img src="_docs/65-add-classification-head-04.png" width="800"/><br><br>
</div>

In [ ]:
print("Last output token:", outputs[:, -1, :])

#### _6.6 Calculating the classification loss and accuracy_

<div>
<img src="_docs/66-calculate-loss-accuracy-01.png" width="800"/><br><br>
<img src="_docs/66-calculate-loss-accuracy-02.png" width="800"/>
</div>

In [ ]:
print(f"Last output token: {outputs[:, -1, :]}\n")

# Obtain the class label.
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print(f"Class probabilities : {probas}")
print(f"Class label         : {label.item()}\n")

# Use of 'softmax' is optional, since the 'argmax' of the logits will be the same as the 'argmax' of the probabilities.
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print(f"Class logits        : {logits}")
print(f"Class label         : {label.item()}")

In [ ]:
# Calculate the classification accuracy.
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    # If 'num_batches' is not specified, iterate over all batches in the data loader. 
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        # Otherwise, limit the number of batches to the specified 'num_batches' or the total number of batches in the data loader, whichever is smaller.
        num_batches = min(num_batches, len(data_loader))

    # Iterate over the batches in the data loader and calculate the number of correct predictions and total examples.
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]     # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (
                (predicted_labels == target_batch).sum().item()
            )
        else:
            break
    return correct_predictions / num_examples 

In [ ]:
# Check the classification accuracies for 10 batches.
torch.manual_seed(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.backends.mps.is_available():
    device = torch.device("mps")
model.to(device)
print(f"Device: {device}\n")

# Calculate and print the accuracies.
train_accuracy = calc_accuracy_loader(
    train_loader, model, device, num_batches=10
)
val_accuracy = calc_accuracy_loader(
    val_loader, model, device, num_batches=10
)
test_accuracy = calc_accuracy_loader(
    test_loader, model, device, num_batches=10
)

print(f"Training accuracy   : {train_accuracy*100:.2f}%")
print(f"Validation accuracy : {val_accuracy*100:.2f}%")
print(f"Test accuracy       : {test_accuracy*100:.2f}%")

In [ ]:
# Calculate the classification loss for a batch of input and target data.
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]     # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    
    return loss

# Calculate the classification loss for a data loader by averaging the loss over a specified number of batches.
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:                                        
        # Ensures number of batches doesn’t exceed batches in data loader.
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break
    
    return total_loss / num_batches

In [ ]:
# Calculate and print the losses for 5 batches.
# Disable gradient tracking for efficiency, model not being trained yet.
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss   : {train_loss:.3f}")
print(f"Validation loss : {val_loss:.3f}")
print(f"Test loss       : {test_loss:.3f}")

#### _6.7 Fine-tuning the model on supervised data_

<div>
<img src="_docs/67-fine-tuning-supervised-data-01.png" width="800"/>
</div>

In [ ]:
# Fine-tune the model to classify spam.
def train_classifier_simple(
    model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter
):
    # Initialize lists to track losses and examples seen.
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # Main training loop.
    for epoch in range(num_epochs):
        # Set model to training mode.
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # Reset loss gradients from the previous batch iteration
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            loss.backward()  # Calculate loss gradients
            optimizer.step()  # Update model weights using loss gradients
            examples_seen += input_batch.shape[0]  # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step.
            # Evaluate the model on training and validation data every 'eval_freq' steps, using 'eval_iter' batches for each evaluation.
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        # Calculate accuracy after each epoch.
        train_accuracy = calc_accuracy_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_accuracy = calc_accuracy_loader(
            val_loader, model, device, num_batches=eval_iter
        )

        print(f"Training accuracy   : {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy : {val_accuracy*100:.2f}%\n")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

# Evaluation function to calculate losses on training and validation data.
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5

# Train the classifier and track losses, accuracies, and examples seen.
train_losses, val_losses, train_accs, val_accs, examples_seen = \
    train_classifier_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=50,
        eval_iter=5
    )

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

<div>
<img src="_docs/67-fine-tuning-supervised-data-02.png" width="800"/>
</div>

In [ ]:
import matplotlib.pyplot as plt

path_plots = 'sms_spam_collection'

# Function to plot training and validation losses against epochs and examples seen, with a secondary x-axis for examples seen.
def plot_values(
        epochs_seen, examples_seen, train_values, val_values,
        label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs.
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(
        epochs_seen, val_values, linestyle="-.",
        label=f"Validation {label}"
    )
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # Create a second x-axis for examples seen.
    ax2 = ax1.twiny()
    ax2.plot(examples_seen, train_values, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # Adjusts layout to make room
    plt.savefig(f"{path_plots}/{label}-plot.pdf")
    plt.show()

# Plot the classification loss.
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses) 

<div>
<img src="_docs/67-fine-tuning-supervised-data-03.png" width="800"/>
</div>

In [ ]:
# Plot the classification accuracy.
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(
    epochs_tensor, examples_seen_tensor, train_accs, val_accs,
    label="accuracy"
)

In [ ]:
# Calculate and print accuracies on the entire dataset.
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy   : {train_accuracy*100:.2f}%")
print(f"Validation accuracy : {val_accuracy*100:.2f}%")
print(f"Test accuracy       : {test_accuracy*100:.2f}%")

#### _6.8 Using the LLM as a spam classifier_

<div>
<img src="_docs/68-llm-as-spam-classifier-01.png" width="800"/>
</div>

In [ ]:
# Define a function to classify new texts.
def classify_review(
    text, model, tokenizer, device, max_length=None, pad_token_id=50256
):
    model.eval()

    # Prepare inputs to the model.
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    # Truncate sequences, if they are too long.
    input_ids = input_ids[:min(
        max_length, supported_context_length
    )]

    # Pad sequences to the longest sequence.
    input_ids += [pad_token_id] * (max_length - len(input_ids))

    input_tensor = torch.tensor(
        input_ids, device=device
    ).unsqueeze(0)  # Add batch dimension

    # Model inference without gradient tracking.
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits of the last output token
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Return the classified result.
    return "spam" if predicted_label == 1 else "not spam"

In [ ]:
# Classify new texts using the fine-tuned model.
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(
    f"'text_1': {classify_review(text_1, model, tokenizer, device, max_length=train_dataset.max_length)}\n"
    f"'text_2': {classify_review(text_2, model, tokenizer, device, max_length=train_dataset.max_length)}"
)

In [ ]:
# Save the fine-tuned model for 'spam-classification' task.
path_model = 'sms_spam_collection'
torch.save(model.state_dict(), f"{path_model}/review_classifier.pth")

# To load the saved model.
model_state_dict = torch.load(f"{path_model}/review_classifier.pth", map_location=device)
model.load_state_dict(model_state_dict)

## Step #9 - Fine-tuning (personal assistant)

### 7 Fine-tuning to follow instructions

<div>
<img src="_docs/70-fine-tuning-follow-instructions.png" width="800"/>
</div>

#### _7.1 Introduction to instruction fine-tuning_

<div>
<img src="_docs/71-instruction-fine-tuning-01.png" width="800"/><br><br>
<img src="_docs/71-instruction-fine-tuning-02.png" width="800"/>
</div>

#### _7.2 Preparing a dataset for supervised instruction fine-tuning_

<div>
<img src="_docs/72-preparing-dataset.png" width="800"/>
</div>

<div class="alert alert-block alert-info">
Check <a href="https://github.com/tatsu-lab/stanford_alpaca" target="_blank">Stanford Alpaca</a> for more details.
</div>

In [ ]:
# Download the dataset.
import json
import os
import urllib

def download_and_load_file(file_path, url, folder_path=None):
    if folder_path is not None:
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
        file_path = os.path.join(folder_path, file_path)
    
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    
    with open(file_path, "r") as file:
        data = json.load(file)
    return data

folder_path = "instruction_finetuning"
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url, folder_path=folder_path)

print(f"Number of entries: {len(data)}\n")
print(f"Example entry:\n{data[50]}\n")
print(f"Another example entry:\n{data[999]}")

In [ ]:
# Implement the prompt formatting function.
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )

    return instruction_text + input_text

# Test the prompt formatting function on a few examples.
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(f"{model_input}{desired_response}\n")

print(
    f"{'-' * 100}\n"
    f"{'-' * 100}\n"
)  # Separator for clarity

model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(f"{model_input}{desired_response}")

In [ ]:
# Partition the dataset.
train_portion = int(len(data) * 0.85)  # Use 85% of the data for training
test_portion = int(len(data) * 0.1)  # Use 10% for testing
val_portion = len(data) - train_portion - test_portion  # Use remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print(f"Training set length     : {len(train_data)}")
print(f"Validation set length   : {len(val_data)}")
print(f"Test set length         : {len(test_data)}")

#### _7.3 Organizing data into training batches_

<div>
<img src="_docs/73-training-batches-01.png" width="800"/><br><br>
<img src="_docs/73-training-batches-02.png" width="800"/><br><br>
<img src="_docs/73-training-batches-03.png" width="800"/>
</div>

In [ ]:
# Implement an instruction dataset class.
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:  # Pretokenize texts
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
# Retrieve the token ID for the end-of-text token used in GPT-2.
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

<div>
<img src="_docs/73-training-batches-04.png" width="800"/>
</div>

<div class="alert alert-block alert-info">

ℹ️ Info:<br>
Step 1: Find the longest sequence in the batch.<br>
Step 2: Pad and prepare inputs.<br>
Step 3: Remove extra padded token added earlier.<br>
Step 4: Convert list of inputs to a tensor and transfer it to target device.

</div>

In [ ]:
# Implement the padding process with a custom collate function.
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Find the longest sequence in the batch.
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst = []

    # Pad and prepare inputs.
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * 
            (batch_max_length - len(new_item))
        )

        # Remove extra padded token added earlier.
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    # Convert list of inputs to a tensor and transfer it to target device.
    inputs_tensor = torch.stack(inputs_lst).to(device)
    
    return inputs_tensor

In [ ]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

<div>
<img src="_docs/73-training-batches-05.png" width="800"/><br><br>
<img src="_docs/73-training-batches-06.png" width="800"/><br><br>
</div>

In [ ]:
# Modify custom_collate_draft_1 and get target tensors as well.
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * 
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

inputs, targets = custom_collate_draft_2(batch)
print(f"Inputs  : {inputs}\n")
print(f"Targets : {targets}")

<div>
<img src="_docs/73-training-batches-07.png" width="800"/><br><br>
<img src="_docs/73-training-batches-08.png" width="800"/>
</div>

In [ ]:
# Implement a custom batch collate function.
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + 
            [pad_token_id] * (batch_max_length - len(new_item))
        )  # Pad sequences to max_length
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        #4 Replace all but the first padding tokens in targets by ignore_index.
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        #5 Optional, truncate to the maximum sequence length.
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    
    return inputs_tensor, targets_tensor

<div class="alert alert-block alert-info">

Check <a href="https://docs.pytorch.org/docs/stable/generated/torch.numel.html" target="_blank">torch.numel</a> for more details.<br>
```
test = torch.tensor([0, 1, 2, 3, 2])

print(test.numel())
print(test.numel() > 1)

mask = test == 2
# indices = torch.nonzero(mask)
indices = torch.nonzero(mask).squeeze()
print(indices)
```
</div>

In [ ]:
inputs, targets = custom_collate_fn(batch)

print(f"Inputs  : {inputs}\n")
print(f"Targets : {targets}")

In [ ]:
logits_1 = torch.tensor(
    [
        [-1.0, 1.0],     # predictions for 1st token
        [-0.5, 1.5]      # predictions for 2nd token
    ]
)
targets_1 = torch.tensor([0, 1]) # Correct token indices to generate
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(f"Loss 1 : {loss_1}\n")

logits_2 = torch.tensor(
    [
        [-1.0, 1.0],     # predictions for 1st token
        [-0.5, 1.5],     # predictions for 2nd token
        [-0.5, 1.5]      # predictions for 3rd token
    ]
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(f"Loss 2 : {loss_2}\n")

targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(f"Loss 3 : {loss_3}\n")
print(f"loss_1 == loss_3: {loss_1 == loss_3}")

<div>
<img src="_docs/73-training-batches-09.png" width="800"/>
</div>

#### _7.4 Creating data loaders for an instruction dataset_

<div>
<img src="_docs/74-creating-dataloaders.png" width="800"/>
</div>

In [ ]:
# Initialize the device variable.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.backends.mps.is_available():
    device = torch.device("mps")
print(f"Device: {device}")

In [ ]:
from functools import partial

# Create a partial function for the custom collate function with specified device and allowed maximum length.
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

<div class="alert alert-block alert-info">

Check <a href="https://docs.python.org/3/library/functools.html#functools.partial" target="_blank">functools.partial</a> for more details.<br>

</div>

In [ ]:
# Initialize the data loaders.
from torch.utils.data import DataLoader

torch.manual_seed(123)
num_workers = 0  # Increase the number, if parallel Python processes are supported by OS.
batch_size = 8

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

#### _7.5 Loading a pretrained LLM_

<div>
<img src="_docs/75-loading-pretrained-llm.png" width="800"/>
</div>

In [ ]:
# Load the pretrained model.
from gpt_download import download_and_load_gpt2
from utils import GPTModel, load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# CHOOSE_MODEL = "gpt2-medium (355M)"
CHOOSE_MODEL = "gpt2-large (774M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
print("Updated BASE_CONFIG:")
for key, value in BASE_CONFIG.items():
    print(f"    {key}: {value}")

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
print(f"\nModel size: {model_size}\n")

settings, params = download_and_load_gpt2(
    model_size=model_size, 
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
# Test the model on an example input.
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(f"input_text -->\n{input_text}")

In [ ]:
from utils import generate, text_to_token_ids, token_ids_to_text

# Ensure input tensor is on the same device as the model.
input_tensor = text_to_token_ids(input_text, tokenizer).to(model.tok_emb.weight.device)

token_ids = generate(
    model=model,
    idx=input_tensor,
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

response_text = generated_text[len(input_text):].strip()
print(f"response_text -->\n{response_text}")

#### _7.6 Fine-tuning the LLM on instruction data_

<div>
<img src="_docs/76-fine-tuning-llm-01.png" width="800"/><br><br>
<img src="_docs/76-fine-tuning-llm-02.png" width="800"/>
</div>

In [ ]:
from utils import calc_loss_loader, train_model_simple  # 5.2 Training an LLM

model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=5
    )
    val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=5
    )

print(f"Training loss   : {train_loss}")
print(f"Validation loss : {val_loss}")

In [ ]:
# Instruction fine-tuning the pretrained LLM.
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.00005, weight_decay=0.1
)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
from utils import plot_losses

# Plot the training and validation losses against epochs and tokens seen.
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

#### _7.7 Extracting and saving responses_

<div>
<img src="_docs/77-extracting-saving-responses-01.png" width="800"/><br><br>
<img src="_docs/77-extracting-saving-responses-02.png" width="800"/>
</div>

In [ ]:
torch.manual_seed(123)

# Iterate over the first three test set samples.
for entry in test_data[:3]:
    input_text = format_input(entry)
    
    # Use the generate function in section 7.5.
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )
    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print(f"\n{'-' * 100}\n")

<div class="alert alert-block alert-success">
Instruction-fine-tuned LLMs such as chatbots are evaluated via multiple approaches:

- Short-answer and multiple-choice benchmarks, such as Measuring Massive Multitask Language Understanding (MMLU; https://arxiv.org/abs/2009.03300), which test the general knowledge of a model.<br>
- Human preference comparison to other LLMs, such as LMSYS chatbot arena (https://arena.lmsys.org).<br>
- Automated conversational benchmarks, where another LLM like GPT-4 is used to evaluate the responses, such as AlpacaEval (https://tatsu-lab.github.io/alpaca_eval/).
</div>

In [ ]:
from tqdm import tqdm
import re

# Generate test set responses.
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )
    test_data[i]["model_response"] = response_text

folder_path = "instruction_finetuning"
with open(f"{folder_path}/instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  # Indent for pretty-printing

print(f"test_data[0]:\n{test_data[0]}")

# Save the fine-tuned model for the 'instruction-following' task.
file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"  # Remove white-spaces and parentheses from file name
torch.save(model.state_dict(), f"{folder_path}/{file_name}")
print(f"Model saved at: {folder_path}/{file_name}")

# Load the saved model.
# model.load_state_dict(torch.load(f"{folder_path}/{file_name}", map_location=device))
# model.to(device)

#### _7.8 Evaluating the fine-tuned LLM_

<div>
<img src="_docs/78-evaluating-llm-01.png" width="800"/><br><br>
<img src="_docs/78-evaluating-llm-02.png" width="800"/>
</div>

<div class="alert alert-block alert-success">
To evaluate test set responses in an automated fashion, we utilize an existing instruction-fine-tuned 8-billion-parameter Llama 3 model developed by Meta AI. This model can be run locally using the open source <a href="https://ollama.com">Ollama</a> application.
</div>
   
<div class="alert alert-block alert-info">
NOTE:

- Ollama is an efficient application for running LLMs on a laptop. It serves as a wrapper around the open source <a href="https://github.com/ggerganov/llama.cpp">llama.cpp</a> library, which implements LLMs in pure `C/C++` to maximize efficiency. 
- However, Ollama is only a tool for generating text using LLMs (inference) and does not support training or fine-tuning LLMs.

__STEPS TO BE TAKEN:__
- _Step #1_ - Install <a href="https://ollama.com">Ollama</a>.
- _Step #2_ - Download `Llama 3` model and verify `Ollama` is functioning correctly, via the command line `terminal`.
    - Either start `Ollama` application or run `ollama serve` in a new `terminal`.
    - With the `Ollama` application or `ollama serve` running in a different terminal, execute the following command on in another new `terminal`. to try out the `8-billion-parameter Llama 3` model:
        ```bash
        ollama run llama3
        ```
- _Step #3_ - Once the model download is complete, we are presented with a command-line interface that allows us to interact with the model. For example, ask the model, `“What do llamas eat?”`
</div>

<div class="alert alert-block alert-warning">

- You can end this `ollama run llama3` session using the input `/bye`.
- However, make sure to keep the `ollama serve` command or the Ollama application running for the remainder of this task.
</div>

In [ ]:
import psutil

# Function to check, if a process with the specified name is running.
def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

# Check if 'ollama' is running, since it is required for the next steps.
ollama_running = check_if_running("ollama")
if not ollama_running:
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
)
print(f"Ollama running: {check_if_running('ollama')}")

<div class="alert alert-block alert-info">

- Load the instruction and response data file created previously and redefine the `format_input` function. Only required, if the earlier `Python` session was already closed or prefer to run rhe remainder of code in a different `Python` session.
    ```python
    import json
    from tqdm import tqdm
    from utils import format_input

    folder_path = "instruction_finetuning"
    file_path = f"{folder_path}/instruction-data-with-response.json"

    # Load the test data with model responses from the JSON file.
    with open(file_path, "r") as file:
        test_data = json.load(file)
    ```
</div>

<div class="alert alert-block alert-info">

- An alternative to the `ollama run` command for interacting with the model is through its `REST API` using `Python`. The `query_model` function shown below demonstrates how to use the `API`.
</div>

In [ ]:
import urllib.request

# Function to query a local Ollama model with a given prompt and return the response.
def query_model(
    prompt, 
    model="llama3", 
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary.
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        # Settings for deterministic responses.
        "options": {
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Convert the dictionary to a JSON-formatted string and encodes it to bytes.
    payload = json.dumps(data).encode("utf-8")

    # Create a request object, setting the method to POST and adding necessary headers.
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )
    request.add_header("Content-Type", "application/json")

    # Send the request and capture the response.
    response_data = ""
    with urllib.request.urlopen(request) as response:
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

In [ ]:
# Querying a local Ollama model.
model = "llama3"
result = query_model("What do Llamas eat?", model)
print(f"result:\n{result}")

<div>
<img src="_docs/78-evaluating-llm-03.png" width="800"/>
</div>

In [ ]:
# Apply the query_model function to score the model responses for the first three entries in the test set.
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
        # f"Respond with the integer number only."  # Modified instruction line to only return the score
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")

In [ ]:
# Function to generate scores for model responses by querying a local Ollama model.
def generate_model_scores(json_data, json_key, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."  # Modified instruction line to only return the score
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

# Evaluate the instruction fine-tuning LLM.
scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}")

#### _7.9 Conclusions_

<div>
<img src="_docs/79-conclusions-01.png" width="800"/><br><br>
<img src="_docs/79-conclusions-02.png" width="800"/>
</div>

<div class="alert alert-block alert-info">
Informational messages here...
</div>

<div class="alert alert-block alert-success">
Success message here...
</div>

<div class="alert alert-block alert-warning">
Warning message here...
</div>

<div class="alert alert-block alert-danger">
Error or critical information here...
</div>